In [45]:
import pandas as pd
import numpy as np
%run Caminhos.ipynb

tabela_clientes = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_clientes_parquet}')
tabela_ind_estrategico = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_ind_estrategico_parquet}')
tabela_alunos = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_alunos_parquet}')
tabela_indicadores_projetos = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_indicadores_projetos_parquet}')
tabela_base_gerada = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_base_gerada_parquet}')
tabela_professor = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_professor_parquet}')
tabela_notas = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_notas_parquet}')
tabela_notas_geral = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_notas_geral_parquet}')
tabela_frequencia_ativo = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_frequencia_ativo_parquet}')
tabela_frequencia_ITA = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_frequencia_ITA_parquet}')
tabela_Tempo_Integral = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_Tempo_Integral_parquet}')

Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/


85 - % de estudantes da rede estadual com frequência regular em 75%

In [46]:
tab_qtd_freq = tabela_frequencia_ativo

## Soma as frequencias dos estudantes inscritos nas turmas ITA/IME
freq_reg = tab_qtd_freq.merge(tabela_clientes[["idMatricula", "Id Escola", "Matricula_2024", "Matricula_2025", "Matricula_2026"]], on='idMatricula', how='left')
freq_reg = freq_reg.drop_duplicates(subset=["idMatricula", "Id Escola"])
freq_reg_aprov = freq_reg[freq_reg['percPresenca'] >= 75]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_apr_24 = freq_reg_aprov[freq_reg_aprov['Matricula_2024'] == 1]
freq_apr_25 = freq_reg_aprov[freq_reg_aprov['Matricula_2025'] == 1]

soma_freq_apr_24 = freq_apr_24.groupby('Id Escola')['idMatricula'].count().reset_index()
soma_freq_apr_25 = freq_apr_25.groupby('Id Escola')['idMatricula'].count().reset_index()

# Adicionar coluna do ano
soma_freq_apr_24['Ano'] = 2024
soma_freq_apr_25['Ano'] = 2025

# Concatenar os DataFrames
soma_freq_apr = pd.concat([soma_freq_apr_24, soma_freq_apr_25])
#display(soma_freq_apr)
'----------------------------------------------------------------------------------------------------------------------------------------'
freq_reg_reprov = freq_reg[freq_reg['percPresenca'] < 75]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_rep_24 = freq_reg_reprov[freq_reg_reprov['Matricula_2024'] == 1]
freq_rep_25 = freq_reg_reprov[freq_reg_reprov['Matricula_2025'] == 1]

soma_freq_rep_24 = freq_rep_24.groupby('Id Escola')['idMatricula'].count().reset_index()
soma_freq_rep_25 = freq_rep_25.groupby('Id Escola')['idMatricula'].count().reset_index()

# Adicionar coluna do ano
soma_freq_rep_24['Ano'] = 2024
soma_freq_rep_25['Ano'] = 2025

# Concatenar os DataFrames
soma_freq_rep = pd.concat([soma_freq_rep_24, soma_freq_rep_25])
#display(soma_freq_rep)
'----------------------------------------------------------------------------------------------------------------------------------------'
est_freq = pd.concat([soma_freq_apr, soma_freq_rep])
#display(est_freq)
'----------------------------------------------------------------------------------------------------------------------------------------'
# Matriculas totais
sum_est_freq = est_freq.groupby(['Id Escola', 'Ano'])['idMatricula'].sum().reset_index()
sum_est_freq = sum_est_freq.rename(columns={'idMatricula': 'Qtde_Total'})
#display(sum_est_freq)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
freq_aprovativa = soma_freq_apr.merge(sum_est_freq, on=['Id Escola', 'Ano'])
freq_aprovativa['% de estudantes da rede estadual com frequência regular em 75%'] = freq_aprovativa['idMatricula'] / freq_aprovativa['Qtde_Total']
freq_aprovativa = freq_aprovativa[["Id Escola", "Ano", "% de estudantes da rede estadual com frequência regular em 75%"]]
display(freq_aprovativa)

,Id Escola,Ano,% de estudantes da rede estadual com frequência regular em 75%
0,4.0,2024,0.666667
1,6.0,2024,0.285714
2,7.0,2024,1.000000
3,8.0,2024,1.000000
4,9.0,2024,0.842105
...,...,...,...
252,834.0,2024,0.701299
253,980.0,2024,1.000000
254,1008.0,2024,0.956522
255,1071.0,2024,0.724138


385 - Indicador - % frequência dos estudantes inscritos nas turmas ITA/IME

In [47]:
tab_freq = tabela_frequencia_ITA

## Qtd aulas com presença dos estudantes inscritos nas turmas ITA/IME
tab_freq = tab_freq.merge(tabela_clientes[["idMatricula", "Id Escola", "Matricula_2024", "Matricula_2025", "Matricula_2026"]], on='idMatricula', how='left')

# Filtrar o DataFrame onde tipoFrequencia = Presença
tab_freq_p = tab_freq[(tab_freq['tipoFrequencia'] == "Presença")]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_24 = tab_freq_p[tab_freq_p['Matricula_2024'] == 1]
freq_25 = tab_freq_p[tab_freq_p['Matricula_2025'] == 1]

soma_freq_24 = freq_24.groupby(['Id Escola', 'Ano'])['idAula'].count().reset_index()
soma_freq_25 = freq_25.groupby(['Id Escola', 'Ano'])['idAula'].count().reset_index()

# Concatenar os DataFrames
soma_freq = pd.concat([soma_freq_24, soma_freq_25])
soma_freq = soma_freq.rename(columns={'idAula': 'Qtde aulas com presença'})
soma_freq['Ano'] = soma_freq['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras
freq_24_total = tab_freq[tab_freq['Matricula_2024'] == 1]
freq_25_total = tab_freq[tab_freq['Matricula_2025'] == 1]

# Contar o número de idMatriculas distintas por Id Estado
estudantes_freq_24  = freq_24_total.groupby(['Id Escola', 'Ano'])['idAula'].count().reset_index()
estudantes_freq_25  = freq_25_total.groupby(['Id Escola', 'Ano'])['idAula'].count().reset_index()

# Concatenar os DataFrames
estudantes_freq = pd.concat([estudantes_freq_24, estudantes_freq_25])
estudantes_freq = estudantes_freq.rename(columns={'idAula': 'Qtde aulas'})
estudantes_freq['Ano'] = estudantes_freq['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_freq = soma_freq.merge(estudantes_freq, on=['Id Escola', 'Ano'])
base_nota_freq['% frequência dos estudantes inscritos nas turmas ITA/IME'] = base_nota_freq['Qtde aulas com presença'] / base_nota_freq['Qtde aulas']
base_nota_freq = base_nota_freq[["Id Escola", "Ano", "% frequência dos estudantes inscritos nas turmas ITA/IME"]]
display(base_nota_freq)

,Id Escola,Ano,% frequência dos estudantes inscritos nas turmas ITA/IME


381 - Indicador - Média nas avaliações nas demais disciplinas

In [48]:
tab_nota_ITA = tabela_notas_geral[["Id Escola", "idMatricula", "Ano", "idTurma", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular

# Filtrar o DataFrame onde turma = ITA/IME
tab_nota_ITA = tab_nota_ITA[(tab_nota_ITA['idTurma'].isin([287816, 287818]))]
# Filtrar o DataFrame onde nomeDisciplina
tab_nota_ITA = tab_nota_ITA[(~tab_nota_ITA['idDisciplina'].isin([2551, 2556, 2557, 2558, 2559, 2560, 2561, 2562, 2563, 2564, 2565, 2566, 2567, 594, 2483, 2554, 595]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_ITA_24 = tab_nota_ITA[tab_nota_ITA['Matricula_2024'] == 1]
nota_ITA_25 = tab_nota_ITA[tab_nota_ITA['Matricula_2025'] == 1]

soma_nota_ITA_24 = nota_ITA_24.groupby(['Id Escola', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_ITA_25 = nota_ITA_25.groupby(['Id Escola', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_ITA = pd.concat([soma_nota_ITA_24, soma_nota_ITA_25])
soma_nota_ITA['Ano'] = soma_nota_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Id Escola
estudantes_ITA_24  = nota_ITA_24.groupby(['Id Escola', 'Ano'])['idMatricula'].count().reset_index()
estudantes_ITA_25  = nota_ITA_25.groupby(['Id Escola', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_ITA = pd.concat([estudantes_ITA_24, estudantes_ITA_25])
estudantes_ITA['Ano'] = estudantes_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_ITA = soma_nota_ITA.merge(estudantes_ITA, on=['Id Escola', 'Ano'])
base_nota_media_ITA['Média nas avaliações nas demais disciplinas'] = base_nota_media_ITA['valorNota'] / base_nota_media_ITA['idMatricula']

base_nota_media_ITA_IME = base_nota_media_ITA.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_ITA_IME = base_nota_media_ITA_IME[(base_nota_media_ITA_IME["Id Escola"] == 3)]
display(base_nota_media_ITA_IME)

,Id Escola,Ano,Média nas avaliações nas demais disciplinas
0,776,2024,7.554592


382 - Indicador - Média nas avaliações das disciplinas especificas ITA/IME

In [49]:
tab_nota_esp_ITA = tabela_notas_geral[["Id Escola", "idMatricula", "idTurma", "Ano", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular
# Filtrar o DataFrame onde turma = ITA/IME
tab_nota_esp_ITA = tab_nota_esp_ITA[(tab_nota_esp_ITA['idTurma'].isin([287816, 287818]))]
# Filtrar o DataFrame onde nomeDisciplina
tab_nota_esp_ITA = tab_nota_esp_ITA[(tab_nota_esp_ITA['idDisciplina'].isin([2551, 2556, 2557, 2558, 2559, 2560, 2561, 2562, 2563, 2564, 2565, 2566, 2567, 594, 2483, 2554, 595]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_esp_ITA_24 = tab_nota_esp_ITA[tab_nota_esp_ITA['Matricula_2024'] == 1]
nota_esp_ITA_25 = tab_nota_esp_ITA[tab_nota_esp_ITA['Matricula_2025'] == 1]

soma_nota_esp_ITA_24 = nota_esp_ITA_24.groupby(['Id Escola', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_esp_ITA_25 = nota_esp_ITA_25.groupby(['Id Escola', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_esp_ITA = pd.concat([soma_nota_esp_ITA_24, soma_nota_esp_ITA_25])
soma_nota_esp_ITA['Ano'] = soma_nota_esp_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Id Estado
estudantes_esp_ITA_24  = nota_esp_ITA_24.groupby(['Id Escola', 'Ano'])['idMatricula'].count().reset_index()
estudantes_esp_ITA_25  = nota_esp_ITA_25.groupby(['Id Escola', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_esp_ITA = pd.concat([estudantes_esp_ITA_24, estudantes_esp_ITA_25])
estudantes_esp_ITA['Ano'] = estudantes_esp_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_esp_ITA = soma_nota_esp_ITA.merge(estudantes_esp_ITA, on=['Id Escola', 'Ano'])
base_nota_media_esp_ITA['Média nas avaliações das disciplinas especificas ITA/IME'] = base_nota_media_esp_ITA['valorNota'] / base_nota_media_esp_ITA['idMatricula']

base_nota_media_esp_ITA_IME = base_nota_media_esp_ITA.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_esp_ITA_IME = base_nota_media_esp_ITA_IME[(base_nota_media_esp_ITA_IME["Id Estado"] == 3)]
display(base_nota_media_esp_ITA_IME)

,Id Escola,Ano,Média nas avaliações das disciplinas especificas ITA/IME
0,776,2024,7.319304


402 - Indicador - Média dos alunos da rede nas disciplinas de línguas estrangeiras

In [50]:
tab_nota_LEst = tabela_notas_geral[["Id Escola", "idMatricula", "Ano", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular Linguas Estrangeiras

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUAS ESTRANGEIRAS
nota_m_LEst = tab_nota_LEst[(tab_nota_LEst['idDisciplina'].isin([2400, 2233, 2232, 11, 10, 9]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_LEst_24 = nota_m_LEst[nota_m_LEst['Matricula_2024'] == 1]
nota_LEst_25 = nota_m_LEst[nota_m_LEst['Matricula_2025'] == 1]

soma_nota_LEst_24 = nota_LEst_24.groupby(['Id Escola', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_LEst_25 = nota_LEst_25.groupby(['Id Escola', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_LEst = pd.concat([soma_nota_LEst_24, soma_nota_LEst_25])
soma_nota_LEst['Ano'] = soma_nota_LEst['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Id Escola
estudantes_LEst_24  = nota_LEst_24.groupby(['Id Escola', 'Ano'])['idMatricula'].count().reset_index()
estudantes_LEst_25  = nota_LEst_25.groupby(['Id Escola', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_LEst = pd.concat([estudantes_LEst_24, estudantes_LEst_25])
soma_nota_LEst['Ano'] = soma_nota_LEst['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_LEst = soma_nota_LEst.merge(estudantes_LEst, on=['Id Escola', 'Ano'])
base_nota_media_LEst['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = base_nota_media_LEst['valorNota'] / base_nota_media_LEst['idMatricula']

base_nota_media_LP = base_nota_media_LEst.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_LP = base_nota_media_LP[(base_nota_media_LP["Id Escola"] == 3)]
display(base_nota_media_LP)

,Id Escola,Ano,Média dos alunos da rede nas disciplinas de línguas estrangeiras
0,3,2024,6.830959
1,4,2024,6.438462
2,6,2024,6.880922
3,7,2024,6.671505
4,8,2024,7.045856
...,...,...,...
544,1049,2024,7.194772
545,1071,2024,6.084074
546,1072,2024,5.550000
547,1089,2024,6.347876


163 - Indicador - % de aprovação dos alunos do Ensino Médio - ok - A Validar

In [51]:
tab_nota_EM_Aprov_geral = tabela_notas_geral[["Id Escola", "idMatricula", "Ano", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025"]]
tab_client_EM_Aprov = tabela_clientes[["idMatricula", "Id Escola", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_EM_Aprov_geral = tab_nota_EM_Aprov_geral.merge(tab_client_EM_Aprov, on=['idMatricula', 'Id Escola', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = LÍNGUA PORTUGUESA
filt_EM = tab_nota_EM_Aprov_geral[
    (tab_nota_EM_Aprov_geral['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_EM_Aprov_geral['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_EM_Aprov_geral['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_EM_24 = filt_EM[filt_EM['Matricula_2024'] == 1]
filt_EM_25 = filt_EM[filt_EM['Matricula_2025'] == 1]

# Agrupando por "Id Escola" e "idMatricula" e calculando a média de "valorNota"
agrupar_media_24 = filt_EM_24.groupby(["Id Escola", "Ano", "idMatricula", "nomeTipoNota"])["valorNota"].mean().reset_index()
agrupar_media_25 = filt_EM_25.groupby(["Id Escola", "Ano", "idMatricula", "nomeTipoNota"])["valorNota"].mean().reset_index()

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [agrupar_media_25, agrupar_media_24]
mat_EM_geral = pd.concat([contar_distintos(df) for df in dataframes])

mat_EM_geral = mat_EM_geral.rename(columns={'idMatricula': 'Qtde_Matriculas'})
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_EM_aprov_geral_24 = agrupar_media_24[((agrupar_media_24['valorNota'] >= 6))]
filt_EM_aprov_geral_25 = agrupar_media_25[((agrupar_media_25['valorNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_EM_aprov_geral_25, filt_EM_aprov_geral_24]
aprovados_EM_geral = pd.concat([contar_distintos(df) for df in dataframes])

aprovados_EM_geral = aprovados_EM_geral.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Id Escola"
EM_geral = mat_EM_geral.merge(aprovados_EM_geral, on=['Id Escola', 'Ano'])
EM_geral['Ano'] = EM_geral['Ano'].astype(int)
"-----------------------------------------------------------------------------------------------------------------"
# % de aprovação dos alunos do Ensino Médio
EM_geral['% de aprovação dos alunos do Ensino Médio'] = EM_geral['Qtde_Matriculas_Aprovadas'] / EM_geral['Qtde_Matriculas']
#EM_geral = mat_EM_geral[mat_EM_geral["Id Escola"] == 3]
EM_geral = EM_geral[['Id Escola', 'Ano', '% de aprovação dos alunos do Ensino Médio']]
display(EM_geral)

,Id Escola,Ano,% de aprovação dos alunos do Ensino Médio
0,3,2024,0.920596
1,4,2024,0.888889
2,6,2024,0.982301
3,7,2024,0.989247
4,8,2024,0.978261
...,...,...,...
526,1071,2024,0.943182
527,1072,2024,0.750000
528,1089,2024,0.954545
529,1102,2024,0.925000


1193 - Indicador - % de estudantes do E.M. com média >= 6 no componente curricular Matemática - ok - Validado

In [52]:
tab_nota_M_Aprovados = tabela_notas[["Id Escola", "idMatricula", "Ano", "mediaNota", "nomeDisciplina", "Matricula_2024", "Matricula_2025"]]
tab_client_M_Aprovados = tabela_clientes[["idMatricula", "Id Escola", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_M_Aprovados = tab_nota_M_Aprovados.merge(tab_client_M_Aprovados, on=['idMatricula', 'Id Escola', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = MATEMÁTICA
filt_M = tab_nota_M_Aprovados[
    (tab_nota_M_Aprovados['nomeDisciplina'] == "MATEMÁTICA") & 
    (tab_nota_M_Aprovados['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_M_Aprovados['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_M_Aprovados['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_M_24 = filt_M[filt_M['Matricula_2024'] == 1]
filt_M_25 = filt_M[filt_M['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_M_25, filt_M_24]
soma_nota_M = pd.concat([contar_distintos(df) for df in dataframes])
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_M_aprovados_24 = filt_M_24[((filt_M_24['mediaNota'] >= 6))]
filt_M_aprovados_25 = filt_M_25[((filt_M_25['mediaNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_M_aprovados_25, filt_M_aprovados_24]
aprovados_M = pd.concat([contar_distintos(df) for df in dataframes])

# Renomear a coluna para identificação clara
aprovados_M = aprovados_M.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Id Estado"
mat_M = soma_nota_M.merge(aprovados_M, on=['Id Escola', 'Ano'])
mat_M['Ano'] = mat_M['Ano'].astype(int)

# Substituir valores NaN por 0 na nova coluna (caso uma escola não tenha nenhuma matrícula com nota >= 6)
mat_M['Qtde_Matriculas_Aprovadas'] = mat_M['Qtde_Matriculas_Aprovadas'].fillna(0)
"-----------------------------------------------------------------------------------------------------------------"
# % de estudantes do E.M. com média >= 6 no componente curricular Matemática
mat_M['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'] = mat_M['Qtde_Matriculas_Aprovadas'] / mat_M['idMatricula']
mat_M = mat_M[["Id Escola", "Ano", "% de estudantes do E.M. com média >= 6 no componente curricular Matemática"]]
display(mat_M)

,Id Escola,Ano,% de estudantes do E.M. com média >= 6 no componente curricular Matemática
0,3,2024,0.804627
1,4,2024,0.649351
2,6,2024,0.990991
3,7,2024,1.000000
4,8,2024,0.933333
...,...,...,...
521,1049,2024,0.876221
522,1071,2024,1.000000
523,1072,2024,0.662162
524,1089,2024,0.666667


1194 - Indicador - % de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa - ok - Validado

In [53]:
tab_nota_LP_Aprovados = tabela_notas[["Id Escola", "idMatricula", "Ano", "mediaNota", "nomeDisciplina", "Matricula_2024", "Matricula_2025"]]
tab_client_LP_Aprovados = tabela_clientes[["idMatricula", "Id Escola", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_LP_Aprovados = tab_nota_LP_Aprovados.merge(tab_client_LP_Aprovados, on=['idMatricula', 'Id Escola', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = LÍNGUA PORTUGUESA
filt_LP = tab_nota_LP_Aprovados[
    (tab_nota_LP_Aprovados['nomeDisciplina'] == "LÍNGUA PORTUGUESA") & 
    (tab_nota_LP_Aprovados['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_LP_Aprovados['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_LP_Aprovados['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_LP_24 = filt_LP[filt_LP['Matricula_2024'] == 1]
filt_LP_25 = filt_LP[filt_LP['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_LP_25, filt_LP_24]
soma_nota_LP = pd.concat([contar_distintos(df) for df in dataframes])
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_LP_aprovados_24 = filt_LP_24[((filt_LP_24['mediaNota'] >= 6))]
filt_LP_aprovados_25 = filt_LP_25[((filt_LP_25['mediaNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_LP_aprovados_25, filt_LP_aprovados_24]
aprovados_nota_LP = pd.concat([contar_distintos(df) for df in dataframes])

aprovados_nota_LP = aprovados_nota_LP.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Id Estado"
nota_LP = soma_nota_LP.merge(aprovados_nota_LP, on=['Id Escola', 'Ano'])
nota_LP['Ano'] = nota_LP['Ano'].astype(int)
"-----------------------------------------------------------------------------------------------------------------"
# % de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa
nota_LP['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'] = nota_LP['Qtde_Matriculas_Aprovadas'] / nota_LP['idMatricula']
#mat_LP = mat_LP[['Id Estado', '% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa']]
nota_LP = nota_LP[["Id Escola", "Ano", "% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa"]]
display(nota_LP)

,Id Escola,Ano,% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa
0,3,2024,0.665198
1,4,2024,0.548387
2,6,2024,0.989848
3,7,2024,1.000000
4,8,2024,0.954545
...,...,...,...
518,1049,2024,0.660131
519,1071,2024,0.941176
520,1072,2024,0.442623
521,1089,2024,0.846154


2237 - Indicador - Nota média dos estudantes da Rede no componente curricular Língua Portuguesa - ok

In [54]:
tab_nota_LP = tabela_notas_geral[["Id Escola", "idMatricula", "Ano", "idDisciplina", "agregada", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular LP

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUA PORTUGUESA
nota_m_LP = tab_nota_LP[(tab_nota_LP['idDisciplina'] == 1)]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_LP_24 = nota_m_LP[nota_m_LP['Matricula_2024'] == 1]
nota_LP_25 = nota_m_LP[nota_m_LP['Matricula_2025'] == 1]

soma_nota_LP_24 = nota_LP_24.groupby(['Id Escola', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_LP_25 = nota_LP_25.groupby(['Id Escola', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_LP = pd.concat([soma_nota_LP_24, soma_nota_LP_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente LP

# Contar o número de idMatriculas distintas por Id Estado
estudantes_LP_24  = nota_LP_24.groupby(['Id Escola', 'Ano'])['idMatricula'].count().reset_index()
estudantes_LP_25  = nota_LP_25.groupby(['Id Escola', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_LP = pd.concat([estudantes_LP_24, estudantes_LP_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
nota_media_LP = soma_nota_LP.merge(estudantes_LP, on=['Id Escola', 'Ano'])
nota_media_LP['Ano'] = nota_media_LP['Ano'].astype(int)
nota_media_LP['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = nota_media_LP['valorNota'] / nota_media_LP['idMatricula']

nota_media_LP = nota_media_LP.drop(['valorNota', 'idMatricula'], axis=1)
#nota_media_LP = nota_media_LP[(nota_media_LP["Id Escola"] == 3)]
display(nota_media_LP)

,Id Escola,Ano,Nota média dos estudantes da Rede no componente curricular Língua Portuguesa
0,3,2024,5.748418
1,4,2024,5.940690
2,6,2024,6.801461
3,7,2024,6.473656
4,8,2024,7.040000
...,...,...,...
546,1049,2024,5.834314
547,1071,2024,6.924638
548,1072,2024,5.455056
549,1089,2024,6.452419


2238 - Indicador - Nota média dos estudantes da Rede no componente curricular Matemática - OK

In [55]:
tab_nota_M = tabela_notas_geral[["Id Escola", "idMatricula", "Ano", "idDisciplina", "agregada", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular M

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUA PORTUGUESA
nota_M = tab_nota_M[(tab_nota_M['idDisciplina'] == 5)]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_M_24 = nota_M[nota_M['Matricula_2024'] == 1]
nota_M_25 = nota_M[nota_M['Matricula_2025'] == 1]

soma_nota_M_24 = nota_M_24.groupby(['Id Escola', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_M_25 = nota_M_25.groupby(['Id Escola', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_M = pd.concat([soma_nota_M_24, soma_nota_M_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente M

# Contar o número de idMatriculas distintas por Id Escola
estudantes_M_24 = nota_M_24.groupby(['Id Escola', 'Ano'])['idMatricula'].count().reset_index()
estudantes_M_25 = nota_M_25.groupby(['Id Escola', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_M = pd.concat([estudantes_M_24, estudantes_M_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_M = soma_nota_M.merge(estudantes_M, on=['Id Escola', 'Ano'])
base_nota_media_M['Ano'] = base_nota_media_M['Ano'].astype(int)

base_nota_media_M['Nota média dos estudantes da Rede no componente curricular Matemática'] = base_nota_media_M['valorNota'] / base_nota_media_M['idMatricula']
base_nota_media_M = base_nota_media_M.drop(['valorNota', 'idMatricula'], axis=1)
display(base_nota_media_M)

,Id Escola,Ano,Nota média dos estudantes da Rede no componente curricular Matemática
0,3,2024,6.296556
1,4,2024,5.667606
2,6,2024,6.921331
3,7,2024,6.286021
4,8,2024,7.068539
...,...,...,...
551,1049,2024,6.084380
552,1071,2024,6.313095
553,1072,2024,5.753077
554,1089,2024,5.937405


2227 - Indicador - Nº de docentes autodeclarados pretos/pardos na rede estadual

2225 - Indicador - Nº de docentes autodeclarados indígenas na rede estadual

2215 - Indicador - N° de estudantes autodeclarados pretos matriculados na rede - ok

In [56]:
tabela_alunos_est = tabela_alunos[["idAluno", "codigoEscolarAluno", "RA", "nome", "sexo", "raca"]]

# Filtrar alunos com raca 'Preta'
tabela_alunos_preta = tabela_alunos_est[tabela_alunos_est['raca'] == 'Preta']

# Mesclar os DataFrames para adicionar as colunas com o cálculo
Matricula_Alunos = tabela_alunos_preta.merge(tabela_clientes[["idAluno", "idMatricula", "Id Escola", "Matricula_2026", "Matricula_2025", "Matricula_2024", "Ano"]], on=['idAluno'], how='inner')

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filter_df_24 = Matricula_Alunos[Matricula_Alunos['Matricula_2024'] == 1]
filter_df_25 = Matricula_Alunos[Matricula_Alunos['Matricula_2025'] == 1]

# Contar os valores distintos na coluna idMatricula por municipio (idMunicipio)
distinct_count_Preto_24_mun = filter_df_24.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_Preto_25_mun = filter_df_25.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames
matricula_aut_pretos = pd.concat([distinct_count_Preto_24_mun, distinct_count_Preto_25_mun])
matricula_aut_pretos.rename(columns={'idMatricula': 'N° de estudantes autodeclarados pretos matriculados na rede'}, inplace=True)
matricula_aut_pretos['Ano'] = matricula_aut_pretos['Ano'].astype(int)
display(matricula_aut_pretos)

,Id Escola,Ano,N° de estudantes autodeclarados pretos matriculados na rede
0,4,2024,2
1,8,2024,3
2,9,2024,8
3,10,2024,1
4,12,2024,2
...,...,...,...
200,980,2024,9
201,1020,2024,3
202,1029,2024,3
203,1071,2024,2


2196 - Indicador - N° de estudantes autodeclarados indígenas matriculados na rede - ok - A Validar

In [57]:
# Filtrar alunos com raca 'Preta'
tabela_alunos_Ind = tabela_alunos_est[tabela_alunos_est['raca'] == 'Indígena']

num_linhas = tabela_alunos_est.shape[0]

# Mesclar os DataFrames para adicionar as colunas com o cálculo
Matricula_Alunos_Ind = tabela_alunos_Ind.merge(tabela_clientes[["idAluno", "idMatricula", "Id Escola", "Matricula_2026", "Matricula_2025", "Matricula_2024", "Ano"]], on=['idAluno'], how='inner')

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filter_Ind_df_24 = Matricula_Alunos_Ind[Matricula_Alunos_Ind['Matricula_2024'] == 1]
filter_Ind_df_25 = Matricula_Alunos_Ind[Matricula_Alunos_Ind['Matricula_2025'] == 1]

# Contar os valores distintos na coluna idMatricula por escola (Id Escola)
idMatricula_Ind_24_escola = filter_Ind_df_24.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()
idMatricula_Ind_25_escola = filter_Ind_df_25.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames
matricula_aut_indigena = pd.concat([idMatricula_Ind_24_escola, idMatricula_Ind_25_escola])
matricula_aut_indigena.rename(columns={'idMatricula': 'N° de estudantes autodeclarados indígenas matriculados na rede'}, inplace=True)
matricula_aut_indigena['Ano'] = matricula_aut_indigena['Ano'].astype(int)
display(matricula_aut_indigena)

,Id Escola,Ano,N° de estudantes autodeclarados indígenas matriculados na rede
0,12,2024,1
1,82,2024,1
2,672,2024,1
3,814,2024,2


153 - Indicador - % de municípios com escolas da rede estadual em tempo integral

In [58]:
# Qtde de municipio com escolas da rede estadual
tabela_clientes_matricula_est = tabela_clientes[["idMatricula", "Id Escola", "Ano", "Id Municipio", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
mun_24 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2024'] == 1]
mun_25 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2025'] == 1]

# Contar os valores distintos na coluna idMatricula por escola (Id Estado)
mun_24_per_mun = mun_24.groupby(['Id Escola', 'Ano'])['Id Municipio'].nunique().reset_index()
mun_25_per_mun = mun_25.groupby(['Id Escola', 'Ano'])['Id Municipio'].nunique().reset_index()

# Concatenar os DataFrames
combined_mun = pd.concat([mun_24_per_mun, mun_25_per_mun])

combined_mun.rename(columns={'Id Municipio': 'Qtde de municipio com escolas da rede estadual'}, inplace=True)
combined_mun['Ano'] = combined_mun['Ano'].astype(int)

In [59]:
# Quantidade de municipio com escolas da rede estadual em tempo integral
tabela_mun_integral = tabela_Tempo_Integral[["Id Escola", "Id Municipio", "Ano", "STATUS_INTEGRAL"]]

filt_status_mun = tabela_mun_integral[tabela_mun_integral['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_24_mun = filt_status_mun[filt_status_mun['Ano'] == 2024]
filt_25_mun = filt_status_mun[filt_status_mun['Ano'] == 2025]

# Somar os valores na coluna Status_integral por matricula (Id Municipio)
sum_24_mun = filt_24_mun.groupby(['Id Escola', 'Ano'])['Id Municipio'].nunique().reset_index()
sum_25_mun = filt_25_mun.groupby(['Id Escola', 'Ano'])['Id Municipio'].nunique().reset_index()

# Concatenar os Dataframes
comb_int_mun = pd.concat([sum_24_mun, sum_25_mun])
comb_int_mun.rename(columns={'Id Municipio': 'Qtde de Municipio com escolas de tempo integral'}, inplace=True)
"----------------------------------------------------------------------------------------------------------------"
# Mesclagem para DF final
resultado_mun = combined_mun.merge(comb_int_mun, on=['Id Escola', 'Ano'], how='left')
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
resultado_mun['% de municípios com escolas da rede estadual em tempo integral'] = resultado_mun['Qtde de Municipio com escolas de tempo integral'] / resultado_mun['Qtde de municipio com escolas da rede estadual']
resultado_mun = resultado_mun[["Id Escola", "% de municípios com escolas da rede estadual em tempo integral", "Ano"]]
display(resultado_mun)

,Id Escola,% de municípios com escolas da rede estadual em tempo integral,Ano
0,4,1.0,2024
1,6,NaN,2024
2,7,NaN,2024
3,8,NaN,2024
4,9,NaN,2024
...,...,...,...
367,1020,NaN,2024
368,1029,NaN,2024
369,1071,NaN,2024
370,1089,NaN,2024


159 - Indicador - Nº total de matrículas da rede estadual de educação

In [60]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filtered_df_24 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2024'] == 1]
filtered_df_23 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2023'] == 1]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filtered_df_23, filtered_df_24]
combined_df = pd.concat([contar_distintos(df) for df in dataframes])

combined_df.rename(columns={'idMatricula': 'Nº total de matrículas da rede estadual de educação'}, inplace=True)
combined_df['Ano'] = combined_df['Ano'].astype(int)
display(combined_df)

,Id Escola,Ano,Nº total de matrículas da rede estadual de educação
0,4,2024,25
1,6,2024,8
2,7,2024,7
3,8,2024,30
4,9,2024,61
...,...,...,...
367,1020,2024,29
368,1029,2024,52
369,1071,2024,51
370,1089,2024,79


2202 - Indicador - % de matrículas de tempo integral no ensino regular, na rede estadual

In [61]:
# Quantidade de matricula de tempo integral por estado
tabela_integral = tabela_Tempo_Integral[["idMatricula", "Id Escola", "Ano", "tempoIntegral"]]
tab_matr = tabela_clientes[["idMatricula", "Id Escola", "Ano", "Id Municipio", "Matricula_2024", "Matricula_2025"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tabela_integral = tabela_integral.merge(tab_matr, on=['idMatricula', 'Id Escola', 'Ano'], how='inner')
filt_status = tabela_integral[tabela_integral['tempoIntegral'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_24 = filt_status[filt_status['Matricula_2024'] == 1]
filt_23 = filt_status[filt_status['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_23, filt_24]
comb_int = pd.concat([contar_distintos(df) for df in dataframes])

comb_int.rename(columns={'idMatricula': 'Qtde de matricula de tempo integral'}, inplace=True)
display(comb_int)
"----------------------------------------------------------------------------------------------------------------"
# Mesclagem para DF final
resultado_final = combined_df.merge(comb_int, on=['Id Escola', 'Ano'], how='inner')
resultado_final['Ano'] = resultado_final['Ano'].astype(int)
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
resultado_final['% de matrículas de tempo integral no ensino regular, na rede estadual'] = resultado_final['Qtde de matricula de tempo integral'] / resultado_final['Nº total de matrículas da rede estadual de educação']
resultado_final = resultado_final[["Id Escola", "% de matrículas de tempo integral no ensino regular, na rede estadual", "Ano"]]
display(resultado_final)

,Id Escola,Ano,Qtde de matricula de tempo integral
0,4,2024.0,25
1,16,2024.0,7
2,27,2024.0,15
3,29,2024.0,251
4,34,2024.0,18
...,...,...,...
90,820,2024.0,24
91,825,2024.0,11
92,980,2024.0,59
93,1008,2024.0,42


,Id Escola,"% de matrículas de tempo integral no ensino regular, na rede estadual",Ano
0,4,1.0,2024
1,16,1.0,2024
2,27,1.0,2024
3,29,1.0,2024
4,34,1.0,2024
...,...,...,...
90,820,1.0,2024
91,825,1.0,2024
92,980,1.0,2024
93,1008,1.0,2024


63 - Indicador - % de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)

In [62]:
# Quantidade de escolas de tempo integral por estado
tabela_integral_EPT = tabela_Tempo_Integral[["Id Escola", "Id Estado", "Ano", "STATUS_INTEGRAL"]]

filt_status_TI = tabela_integral_EPT[tabela_integral_EPT['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_25_TI = filt_status_TI[filt_status_TI['Ano'] == 2025]
filt_24_TI = filt_status_TI[filt_status_TI['Ano'] == 2024]
filt_23_TI = filt_status_TI[filt_status_TI['Ano'] == 2023]
filt_22_TI = filt_status_TI[filt_status_TI['Ano'] == 2022]

base_25_TI = filt_25_TI[["Id Escola", "Id Estado"]]
base_24_TI = filt_24_TI[["Id Escola", "Id Estado"]]
base_23_TI = filt_23_TI[["Id Escola", "Id Estado"]]
base_22_TI = filt_22_TI[["Id Escola", "Id Estado"]]

# Remover duplicatas do DataFrame, mantendo apenas a primeira ocorrência
base_25_TI = base_25_TI.drop_duplicates(keep='first')
base_24_TI = base_24_TI.drop_duplicates(keep='first')
base_23_TI = base_23_TI.drop_duplicates(keep='first')
base_22_TI = base_22_TI.drop_duplicates(keep='first')

base_23_TI = pd.concat([base_23_TI, base_22_TI])
base_23_TI['Ano'] = 2023
base_24_TI = pd.concat([base_24_TI, base_23_TI])
base_24_TI['Ano'] = 2024

# Concatenar os DataFrames
df_combinado_g = pd.concat([base_24_TI, base_23_TI], ignore_index=True)
"----------------------------------------------------------------------------------------------------------------"

# Somar os valores na coluna Status_integral por matricula (Id Matricula)
base_25_TI['Qtde de escolas de tempo integral'] = 1
base_25_TI['Ano'] = 2025
base_24_TI['Qtde de escolas de tempo integral'] = 1
base_24_TI['Ano'] = 2024
base_23_TI['Qtde de escolas de tempo integral'] = 1
base_23_TI['Ano'] = 2023
base_22_TI['Qtde de escolas de tempo integral'] = 1
base_22_TI['Ano'] = 2022

# Concatenar os DataFrames
df_combinado = pd.concat([base_24_TI, base_23_TI], ignore_index=True)
# Substitua 'Nome_da_Coluna' pelo nome da coluna que deseja remover
df_combinado = df_combinado.drop('Id Estado', axis=1)

# Exibir o DataFrame combinado
display(df_combinado)
#EPT['Qtde de escolas de tempo integral'] = 1

,Id Escola,Ano,Qtde de escolas de tempo integral
0,433,2024,1
1,607,2024,1
2,563,2024,1
3,549,2024,1
4,553,2024,1
...,...,...,...
547,427,2023,1
548,733,2023,1
549,456,2023,1
550,611,2023,1


In [63]:
# Tabela filtrada
tabela_clientes_TI_EPT = tabela_clientes[["Id Escola", "Id Estado", "Matricula_2023", "Matricula_2024", "EtapaAgregada", "ordem"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filtered_TI_EPT_24 = tabela_clientes_TI_EPT[(tabela_clientes_TI_EPT['Matricula_2024'] == 1) & (tabela_clientes_TI_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_TI_EPT_24 = filtered_TI_EPT_24[["Id Escola", "Id Estado"]]

filtered_TI_EPT_24 = filtered_TI_EPT_24.drop_duplicates(keep='first')

filtered_TI_EPT_23 = tabela_clientes_TI_EPT[(tabela_clientes_TI_EPT['Matricula_2023'] == 1) & (tabela_clientes_TI_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_TI_EPT_23 = filtered_TI_EPT_23[["Id Escola", "Id Estado"]]

filtered_TI_EPT_23 = filtered_TI_EPT_23.drop_duplicates(keep='first')

# Concatenar os DataFrames
df_c_EPT = pd.concat([filtered_TI_EPT_24, filtered_TI_EPT_23], ignore_index=True)
#display(df_c_EPT)
"----------------------------------------------------------------------------------------------------------------"
# Relação inner join para DF final
TI_EPT = df_combinado_g.merge(df_c_EPT, how='inner')
TI_EPT = TI_EPT.drop_duplicates(keep='first')
#display(TI_EPT)
"----------------------------------------------------------------------------------------------------------------"
TI_EPT_24 = TI_EPT[TI_EPT['Ano'] == 2024]
#TI_EPT_24 = TI_EPT_24[["Id Estado", "Id Escola", "Ano"]]
TI_EPT_23 = TI_EPT[TI_EPT['Ano'] == 2023]

TI_EPT_24['Qtde de escolas EPT em TI'] = 1
TI_EPT_23['Qtde de escolas EPT em TI'] = 1

TI_TI_EPT = pd.concat([TI_EPT_24, TI_EPT_23])
# Substitua 'Nome_da_Coluna' pelo nome da coluna que deseja remover
TI_TI_EPT = TI_TI_EPT.drop('Id Estado', axis=1)

#display(TI_TI_EPT)

result_TI_EPT = TI_TI_EPT.merge(df_combinado, how='inner')
display(result_TI_EPT)
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
result_TI_EPT['% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)'] = result_TI_EPT['Qtde de escolas EPT em TI'] / result_TI_EPT['Qtde de escolas de tempo integral']
result_TI_EPT = result_TI_EPT[["Id Escola", "% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)", "Ano"]]
display(result_TI_EPT)
result_TI_EPT.info()

C:\Users\VeryPC\AppData\Local\Temp\ipykernel_10144\1855028072.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TI_EPT_24['Qtde de escolas EPT em TI'] = 1
C:\Users\VeryPC\AppData\Local\Temp\ipykernel_10144\1855028072.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TI_EPT_23['Qtde de escolas EPT em TI'] = 1


,Id Escola,Ano,Qtde de escolas EPT em TI,Qtde de escolas de tempo integral
0,599,2024,1,1
1,820,2024,1,1
2,778,2024,1,1
3,286,2024,1,1
4,544,2024,1,1
5,528,2024,1,1
6,30,2024,1,1
7,681,2024,1,1
8,583,2024,1,1
9,642,2024,1,1


,Id Escola,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),Ano
0,599,1.0,2024
1,820,1.0,2024
2,778,1.0,2024
3,286,1.0,2024
4,544,1.0,2024
5,528,1.0,2024
6,30,1.0,2024
7,681,1.0,2024
8,583,1.0,2024
9,642,1.0,2024


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 3 columns):
 #   Column                                                                                      Non-Null Count  Dtype  
---  ------                                                                                      --------------  -----  
 0   Id Escola                                                                                   16 non-null     int64  
 1   % de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)  16 non-null     float64
 2   Ano                                                                                         16 non-null     int64  
dtypes: float64(1), int64(2)
memory usage: 516.0 bytes


2213 - Indicador - Nº de matrículas EPT - ok - A Validar

In [64]:
# Tabela filtrada
tabela_clientes_m_ept = tabela_clientes[["idMatricula", "Id Escola", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "ordem"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filtered_df_EPTT_25 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2025'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_25 = filtered_df_EPTT_25[["idMatricula", "Id Escola", "Ano", "Matricula_2025"]]
filtered_df_EPTT_24 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2024'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_24 = filtered_df_EPTT_24[["idMatricula", "Id Escola", "Ano", "Matricula_2024"]]
filtered_df_EPTT_23 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2023'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_23 = filtered_df_EPTT_23[["idMatricula", "Id Escola", "Matricula_2023", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "Profissional"
distinct_count_n_EPT_25 = filtered_df_EPTT_25.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_n_EPT_24 = filtered_df_EPTT_24.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_n_EPT_23 = filtered_df_EPTT_23.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem profissional por escola
combined_df_profissional = pd.concat([distinct_count_n_EPT_25, distinct_count_n_EPT_24, distinct_count_n_EPT_23])
combined_df_profissional.rename(columns={'idMatricula': 'Nº de matrículas EPT'}, inplace=True)
combined_df_profissional['Ano'] = combined_df_profissional['Ano'].astype(int)
display(combined_df_profissional)

,Id Escola,Ano,Nº de matrículas EPT
0,30,2024,1
1,102,2024,1
2,191,2024,1
3,213,2024,1
4,231,2024,1
5,286,2024,2
6,311,2024,2
7,316,2024,1
8,473,2024,1
9,528,2024,1


72 - Indicador - Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC) - ok - A Validar

In [65]:
# Tabela filtrada
tabela_clientes_prof = tabela_clientes[["idMatricula", "Id Escola", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "EtapaAgregada", "NomeCurso", "NomeEtapa"]]

# Lista de textos a serem contados
cursos_filtrar = [
    "ADMINISTRAÇÃO",
    "CURSO TÉCNICO EM ARTESANATO COM ÊNFASE EM EMPREENDEDORISMO",
    "CONTROLE AMBIENTAL",
    "CURSO TÉCNICO EM DESENVOLVIMENTO DE SISTEMAS",
    "CURSO TÉCNICO EM ELETROTÉCNICA COM ÊNFASE EM ELETROMECÂNICA",
    "CURSO TÉCNICO EM ELETROTÉCNICA COM ÊNFASE EM HIDROGÊNIO VERDE",
    "CURSO TÉCNICO EM GUIA DE TURISMO",
    "CURSO TÉCNICO EM MARKETING DIGITAL",
    "CURSO TÉCNICO EM MINERAÇÃO",
    "PRODUÇÃO DE ÁUDIO E VÍDEO",
    "CURSO TÉCNICO EM PROGRAMAÇÃO DE JOGOS DIGITAIS",
    "CURSO TÉCNICO EM SISTEMAS DE ENERGIA RENOVÁVEL"
]

# Criar uma expressão regular que combine qualquer um dos textos
pattern = '|'.join(cursos_filtrar)

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
df_SEDUCTEC_25 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2025'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_25 = df_SEDUCTEC_25[["idMatricula", "Id Escola", "Ano", "NomeCurso", "NomeEtapa", "Matricula_2025"]]
df_SEDUCTEC_24 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2024'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_24 = df_SEDUCTEC_24[["idMatricula", "Id Escola", "Ano", "NomeCurso", "NomeEtapa", "Matricula_2024"]]
df_SEDUCTEC_23 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2023'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_23 = df_SEDUCTEC_23[["idMatricula", "Id Escola", "Ano", "NomeCurso", "NomeEtapa", "Matricula_2023"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "Profissional"
count_SEDUCTEC_25 = df_SEDUCTEC_25.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()
count_SEDUCTEC_24 = df_SEDUCTEC_24.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()
count_SEDUCTEC_23 = df_SEDUCTEC_23.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem profissional por escola
combined_df_SEDUCTEC = pd.concat([count_SEDUCTEC_25, count_SEDUCTEC_24, count_SEDUCTEC_23])
combined_df_SEDUCTEC['Ano'] = combined_df_SEDUCTEC['Ano'].astype(int)
combined_df_SEDUCTEC.rename(columns={'idMatricula': 'Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'}, inplace=True)
display(combined_df_SEDUCTEC)

,Id Escola,Ano,Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)
0,6,2024,1
1,7,2024,1
2,8,2024,30
3,9,2024,38
4,11,2024,29
...,...,...,...
186,833,2024,38
187,834,2024,43
188,1008,2024,12
189,1020,2024,28


67 - Indicador - Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais - ok - A validar

In [66]:
# Tabela filtrada
tabela_clientes_EJA = tabela_clientes[["idMatricula", "Id Escola", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filtered_df_EJAT_25 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2025'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_25 = filtered_df_EJAT_25[["idMatricula", "Id Escola", "Ano", "Matricula_2025"]]
filtered_df_EJAT_24 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2024'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_24 = filtered_df_EJAT_24[["idMatricula", "Id Escola", "Ano", "Matricula_2024"]]
filtered_df_EJAT_23 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2023'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_23 = filtered_df_EJAT_23[["idMatricula", "Id Escola", "Ano", "Matricula_2023"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"
EJAT_25 = filtered_df_EJAT_25.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()
EJAT_24 = filtered_df_EJAT_24.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()
EJAT_23 = filtered_df_EJAT_23.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem EJA por escola
combined_df_EJA = pd.concat([EJAT_25, EJAT_24, EJAT_23])
combined_df_EJA.rename(columns={'idMatricula': 'Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'}, inplace=True)
combined_df_EJA['Ano'] = combined_df_EJA['Ano'].astype(int)
display(combined_df_EJA)

,Id Escola,Ano,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais
0,4,2024,15
1,7,2024,7
2,10,2024,5
3,12,2024,19
4,23,2024,30
...,...,...,...
146,833,2024,38
147,834,2024,123
148,1029,2024,52
149,1071,2024,51


2192 - Indicador - Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT) - ok - A Validar

In [67]:
# Tabela filtrada
tabela_clientes_EJA_EPT = tabela_clientes[["Id Estado", "Id Escola", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filtered_m_EJA_EPT_25 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2025'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_25 = filtered_m_EJA_EPT_25[["Id Escola", "Id Estado", "Ano"]]
filtered_m_EJA_EPT_24 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2024'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_24 = filtered_m_EJA_EPT_24[["Id Escola", "Id Estado", "Ano"]]
filtered_m_EJA_EPT_23 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2023'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_23 = filtered_m_EJA_EPT_23[["Id Escola", "Id Estado", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA" 'matriculaAtiva'
filtered_m_EJA_EPT_25['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'] = 1
filtered_m_EJA_EPT_24['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'] = 1
filtered_m_EJA_EPT_23['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'] = 1

# Concatenar os DataFrames de contagem por escola
EJA_EPT = pd.concat([filtered_m_EJA_EPT_25, filtered_m_EJA_EPT_24, filtered_m_EJA_EPT_23])
# Substitua 'Nome_da_Coluna' pelo nome da coluna que deseja remover
EJA_EPT = EJA_EPT.drop('Id Estado', axis=1)
EJA_EPT = EJA_EPT.drop_duplicates(keep='first')

EJA_EPT['Ano'] = EJA_EPT['Ano'].astype(int)
display(EJA_EPT)

,Id Escola,Ano,Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)


65 - Indicador - Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)

In [68]:
# Tabela filtrada
tabela_clientes_EPT = tabela_clientes[["Id Estado", "Id Escola", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_df_EPT_25 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2025'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_25 = filt_df_EPT_25[["Id Estado", "Ano", "Id Escola"]]
filt_df_EPT_24 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2024'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_24 = filt_df_EPT_24[["Id Estado", "Ano", "Id Escola"]]
filt_df_EPT_23 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2023'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_23 = filt_df_EPT_23[["Id Estado", "Ano", "Id Escola"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_df_EPT_25['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'] = 1
filt_df_EPT_24['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'] = 1
filt_df_EPT_23['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'] = 1

EPT = pd.concat([filt_df_EPT_25, filt_df_EPT_24, filt_df_EPT_23])
EPT = EPT.drop('Id Estado', axis=1)
EPT = EPT.drop_duplicates(keep='first')

EPT['Ano'] = EPT['Ano'].astype(int)
display(EPT)

,Ano,Id Escola,Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)
382640,2024,191,1
386586,2024,213,1
386587,2024,311,1
396153,2024,286,1
460984,2024,738,1
475794,2024,820,1
495420,2024,102,1
496440,2024,544,1
499263,2024,231,1
504027,2024,583,1


1191 - Indicador - % de escolas da zona rural com oferta de EPT

In [69]:
# Tabela filtrada
tab_rural_EPT_tot = tabela_clientes[["Id Estado", "Id Escola", "Ano", "ativa", "ordem", "localizacao", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]
tab_rural_EPT_tot = tab_rural_EPT_tot[(tab_rural_EPT_tot['ativa'] == 1) & (tab_rural_EPT_tot['localizacao'].str.contains("Rural"))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_rural_EPTT_25 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2025'] == 1)]
filt_rural_EPTT_25 = filt_rural_EPTT_25[["Id Estado", "Id Escola", "Ano"]]
filt_rural_EPTT_24 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2024'] == 1)]
filt_rural_EPTT_24 = filt_rural_EPTT_24[["Id Estado", "Id Escola", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_rural_EPTT_25['Nº de escolas da zona rural'] = 1
filt_rural_EPTT_24['Nº de escolas da zona rural'] = 1

rural_EPTT = pd.concat([filt_rural_EPTT_25, filt_rural_EPTT_24])

# Remover duplicatas com base nas colunas 'Id Estado' e 'Ano'
rural_EPTT = rural_EPTT.drop('Id Estado', axis=1)
rural_EPTT = rural_EPTT.drop_duplicates(keep='first')

rural_EPTT['Ano'] = rural_EPTT['Ano'].astype(int)
display(rural_EPTT)

,Id Escola,Ano,Nº de escolas da zona rural
360082,517,2024,1
395739,283,2024,1
425377,504,2024,1
426715,513,2024,1
439068,596,2024,1
455694,703,2024,1
479168,980,2024,1
529861,404,2024,1
534501,331,2024,1
535183,669,2024,1


In [70]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_rural_EPT_25 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2025'] == 1) & (tab_rural_EPT_tot['ordem'].isin([6, 7, 8, 13]))]
filt_rural_EPT_25 = filt_rural_EPT_25[["Id Estado", "Id Escola", "Ano"]]
filt_rural_EPT_24 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2024'] == 1) & (tab_rural_EPT_tot['ordem'].isin([6, 7, 8, 13]))]
filt_rural_EPT_24 = filt_rural_EPT_24[["Id Estado", "Id Escola", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_rural_EPT_25['Nº de escolas da zona rural com oferta de EPT'] = 1
filt_rural_EPT_24['Nº de escolas da zona rural com oferta de EPT'] = 1

rural_EPT = pd.concat([filt_rural_EPT_24, filt_rural_EPT_25])

# Remover duplicatas com base nas colunas 'Id Estado' e 'Ano'
rural_EPT = rural_EPT.drop('Id Estado', axis=1)
rural_EPT = rural_EPT.drop_duplicates(keep='first')

rural_EPT['Ano'] = rural_EPT['Ano'].astype(int)
rural_final = rural_EPTT.merge(rural_EPT, on=['Id Escola', 'Ano'], how='left')

rural_final['% de escolas da zona rural com oferta de EPT'] = rural_final['Nº de escolas da zona rural com oferta de EPT'] / rural_final['Nº de escolas da zona rural']
rural_final = rural_final[["Id Escola", "% de escolas da zona rural com oferta de EPT", "Ano"]]
display(rural_final)

,Id Escola,% de escolas da zona rural com oferta de EPT,Ano
0,517,NaN,2024
1,283,NaN,2024
2,504,NaN,2024
3,513,NaN,2024
4,596,NaN,2024
5,703,NaN,2024
6,980,NaN,2024
7,404,NaN,2024
8,331,NaN,2024
9,669,NaN,2024


2275 - Indicador - % de escolas da zona rural com oferta de Tempo Integral

In [71]:
# Quantidade de escolas da zona rural com oferta de tempo integral por estado
tabela_integral_Ru = tabela_Tempo_Integral[["Id Escola", "Id Estado", "localizacao", "ativa", "Ano", "STATUS_INTEGRAL"]]
tabela_integral_Ru = tabela_integral_Ru[(tabela_integral_Ru['ativa'] == 1) & (tabela_integral_Ru['localizacao'].str.contains("Rural"))]

filt_status_Ru = tabela_integral_Ru[tabela_integral_Ru['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_25_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2025]
filt_24_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2024]
filt_23_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2023]
filt_22_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2022]

base_25_Ru = filt_25_Ru[["Id Escola", "Id Estado"]]
base_24_Ru = filt_24_Ru[["Id Escola", "Id Estado"]]
base_23_Ru = filt_23_Ru[["Id Escola", "Id Estado"]]
base_22_Ru = filt_22_Ru[["Id Escola", "Id Estado"]]

# Remover duplicatas do DataFrame, mantendo apenas a primeira ocorrência
base_25_Ru = base_25_Ru.drop_duplicates(keep='first')
base_24_Ru = base_24_Ru.drop_duplicates(keep='first')
base_23_Ru = base_23_Ru.drop_duplicates(keep='first')
base_22_Ru = base_22_Ru.drop_duplicates(keep='first')

base_23_Ru = pd.concat([base_23_Ru, base_22_Ru])
base_23_Ru['Ano'] = 2023
base_24_Ru = pd.concat([base_24_Ru, base_23_Ru])
base_24_Ru['Ano'] = 2024

# Concatenar os DataFrames
df_combinado_Ru = pd.concat([base_24_Ru, base_23_Ru], ignore_index=True)
"----------------------------------------------------------------------------------------------------------------"

# Somar os valores na coluna Status_integral por matricula (Id Matricula)
base_25_Ru = base_25_Ru.groupby('Id Estado')['Id Escola'].nunique().reset_index()
base_25_Ru['Ano'] = 2025
base_24_Ru = base_24_Ru.groupby('Id Estado')['Id Escola'].nunique().reset_index()
base_24_Ru['Ano'] = 2024
base_23_Ru = base_23_Ru.groupby('Id Estado')['Id Escola'].nunique().reset_index()
base_23_Ru['Ano'] = 2023

# Concatenar os DataFrames
df_combinad = pd.concat([base_24_Ru], ignore_index=True)
df_combinad.rename(columns={'Id Escola': 'Qtde de escolas da zona rural com oferta de Tempo Integral'}, inplace=True)

# Exibir o DataFrame combinado
display(df_combinad)

,Id Estado,Qtde de escolas da zona rural com oferta de Tempo Integral,Ano
0,22,12,2024


2212 - Indicador - Nº de matrículas de EJA integrado ao EPT - ok - A Validar

In [72]:
# Tabela filtrada
tabela_clientes_inte = tabela_clientes[["idMatricula", "Id Escola", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]
tabela_clientes_inte = tabela_clientes_inte[(tabela_clientes_inte['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_inte['ativa'] == 1) & (tabela_clientes_inte['ordem'].isin([6, 7, 8, 13]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filt_df_EJA_EPT_25 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2025'] == 1)]
filt_df_EJA_EPT_25 = filt_df_EJA_EPT_25[["idMatricula", "Id Escola", "Ano"]]
filt_df_EJA_EPT_24 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2024'] == 1)]
filt_df_EJA_EPT_24 = filt_df_EJA_EPT_24[["idMatricula", "Id Escola", "Ano"]]
filt_df_EJA_EPT_23 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2023'] == 1)]
filt_df_EJA_EPT_23 = filt_df_EJA_EPT_23[["idMatricula", "Id Escola", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
dist_EJA_EPT_25 = filt_df_EJA_EPT_25.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()
dist_EJA_EPT_24 = filt_df_EJA_EPT_24.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()
dist_EJA_EPT_23 = filt_df_EJA_EPT_23.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem por escola
combined_df_EJA_EPT = pd.concat([dist_EJA_EPT_25, dist_EJA_EPT_24, dist_EJA_EPT_23])
combined_df_EJA_EPT.rename(columns={'idMatricula': 'Nº de matrículas de EJA integrado ao EPT'}, inplace=True)
combined_df_EJA_EPT['Ano'] = combined_df_EJA_EPT['Ano'].astype(int)
display(combined_df_EJA_EPT)

,Id Escola,Ano,Nº de matrículas de EJA integrado ao EPT


2246 - Indicador - Nº de matrículas do Ensino Médio - ok - A VALIDAR

In [73]:
# Tabela filtrada
tabela_clientes_m = tabela_clientes[["idMatricula", "Id Escola", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Filtro 'Ensino Médio'
tabela_clientes_m = tabela_clientes_m[
    (tabela_clientes_m['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tabela_clientes_m['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tabela_clientes_m['grupo'].str.contains('Ensino', regex=False))
]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filt_m_25 = tabela_clientes_m[(tabela_clientes_m['Matricula_2025'] == 1)]
filt_m_25 = filt_m_25[["idMatricula", "Id Escola", "Ano", "STATUS_INTEGRAL"]]
filt_m_24 = tabela_clientes_m[(tabela_clientes_m['Matricula_2024'] == 1)]
filt_m_24 = filt_m_24[["idMatricula", "Id Escola", "Ano", "STATUS_INTEGRAL"]]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_m_25, filt_m_24]
combined_df_EM = pd.concat([contar_distintos(df) for df in dataframes])

combined_df_EM.rename(columns={'idMatricula': 'Nº de matrículas do Ensino Médio'}, inplace=True)
combined_df_EM['Ano'] = combined_df_EM['Ano'].astype(int)

display(combined_df_EM)

,Id Escola,Ano,Nº de matrículas do Ensino Médio
0,6,2024,8
1,11,2024,1
2,29,2024,12
3,30,2024,1
4,34,2024,1
...,...,...,...
118,824,2024,2
119,825,2024,6
120,834,2024,1
121,1020,2024,1


61 - Indicador - % de matrículas do ensino médio em tempo integral

In [74]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filt_m_24 = tabela_clientes_m[(tabela_clientes_m['Matricula_2024'] == 1) & (tabela_clientes_m['STATUS_INTEGRAL'] == 1)]
filt_m_24 = filt_m_24[["idMatricula", "Id Escola", "Ano"]]
filt_m_25 = tabela_clientes_m[(tabela_clientes_m['Matricula_2025'] == 1) & (tabela_clientes_m['STATUS_INTEGRAL'] == 1)]
filt_m_25 = filt_m_25[["idMatricula", "Id Escola", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_m_25, filt_m_24]
combined_TI_EM = pd.concat([contar_distintos(df) for df in dataframes])

combined_TI_EM.rename(columns={'idMatricula': 'Nº de matrículas do Ensino Médio em tempo integral'}, inplace=True)
"--------------------------------------------------------------------------------------------------------------------------------------------------------------------"

result_TI_EM = combined_df_EM.merge(combined_TI_EM, on=['Id Escola', 'Ano'], how='left')
result_TI_EM['Ano'] = result_TI_EM['Ano'].astype(int)
"--------------------------------------------------------------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
result_TI_EM['% de matrículas do ensino médio em tempo integral'] = result_TI_EM['Nº de matrículas do Ensino Médio em tempo integral'] / result_TI_EM['Nº de matrículas do Ensino Médio']
result_TI_EM = result_TI_EM[["Id Escola", "% de matrículas do ensino médio em tempo integral", "Ano"]]

display(result_TI_EM)

,Id Escola,% de matrículas do ensino médio em tempo integral,Ano
0,6,NaN,2024
1,11,1.0,2024
2,29,NaN,2024
3,30,1.0,2024
4,34,1.0,2024
...,...,...,...
118,824,1.0,2024
119,825,1.0,2024
120,834,NaN,2024
121,1020,1.0,2024


2230 - Indicador - Nº de escolas com oferta de tempo integral no ensino médio

In [75]:
# N° De escolas com ensino médio
tabela_clientes_e = tabela_clientes[["Id Estado", "Id Escola", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL", "tempoIntegralEscola"]]

# Filtro 'Ensino Médio'
tabela_clientes_e = tabela_clientes_e[
    (tabela_clientes_e['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tabela_clientes_e['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tabela_clientes_e['grupo'].str.contains('Ensino', regex=False)) &
    (tabela_clientes_e['tempoIntegralEscola'] == 1)
]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filtered_e_25 = tabela_clientes_e[(tabela_clientes_e['Matricula_2025'] == 1)]
filtered_e_25 = filtered_e_25[["Id Escola", "Ano"]]
filtered_e_24 = tabela_clientes_e[(tabela_clientes_e['Matricula_2024'] == 1)]
filtered_e_24 = filtered_e_24[["Id Escola", "Ano"]]
filtered_e_23 = tabela_clientes_e[(tabela_clientes_e['Matricula_2023'] == 1)]
filtered_e_23 = filtered_e_23[["Id Escola", "Ano"]]

combined_e_EM = pd.concat([filtered_e_25, filtered_e_24, filtered_e_23])
combined_e_EM['Nº de escolas com oferta de tempo integral no ensino médio'] = 1
combined_e_EM = combined_e_EM.drop_duplicates(keep='first')
combined_e_EM['Ano'] = combined_e_EM['Ano'].astype(int)
display(combined_e_EM)

,Id Escola,Ano,Nº de escolas com oferta de tempo integral no ensino médio
360081,55,2024,1
362478,69,2024,1
362479,343,2024,1
364098,78,2024,1
374548,147,2024,1
...,...,...,...
615861,488,2024,1
626366,280,2024,1
666140,34,2024,1
684206,386,2024,1


2208 - Indicador - Nº de cursos EPT ofertados - ok - A Validar

In [76]:
# Tabela filtrada
tabela_clientes_ept_of = tabela_clientes[["idMatricula", "Id Escola", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeCurso", "ordem"]]
tabela_clientes_ept_of = tabela_clientes_ept_of[(tabela_clientes_ept_of['ordem'].isin([6, 7, 8, 13]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filt_df_EPT_curso_25 = tabela_clientes_ept_of[(tabela_clientes_ept_of['Matricula_2025'] == 1)]
filt_df_EPT_curso_25 = filt_df_EPT_curso_25[["Id Escola", "Ano", "NomeCurso"]]
filt_df_EPT_curso_24 = tabela_clientes_ept_of[(tabela_clientes_ept_of['Matricula_2024'] == 1)]
filt_df_EPT_curso_24 = filt_df_EPT_curso_24[["Id Escola", "Ano", "NomeCurso"]]
filt_df_EPT_curso_23 = tabela_clientes_ept_of[(tabela_clientes_ept_of['Matricula_2023'] == 1)]
filt_df_EPT_curso_23 = filt_df_EPT_curso_23[["Id Escola", "Ano", "NomeCurso"]]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['NomeCurso'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_df_EPT_curso_25, filt_df_EPT_curso_24, filt_df_EPT_curso_23]
combined_df_cursos_profissional = pd.concat([contar_distintos(df) for df in dataframes])

combined_df_cursos_profissional.rename(columns={'NomeCurso': 'Nº de cursos EPT ofertados'}, inplace=True)
combined_df_cursos_profissional['Ano'] = combined_df_cursos_profissional['Ano'].astype(int)
display(combined_df_cursos_profissional)

,Id Escola,Ano,Nº de cursos EPT ofertados
0,30,2024,1
1,102,2024,1
2,191,2024,1
3,213,2024,1
4,231,2024,1
5,286,2024,1
6,311,2024,1
7,316,2024,1
8,473,2024,1
9,528,2024,1


2232 - Indicador - Nº de matrículas da educação especial (AEE) - ok - A Validar

In [77]:
# Tabela filtrada
tabela_clientes_AEE = tabela_clientes[["idMatricula", "Id Escola", "Ano", "NomeCurso", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]
tabela_clientes_AEE = tabela_clientes_AEE[(tabela_clientes_AEE['NomeCurso'] == "AEE")]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filt_df_AEE_25 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2025'] == 1)]
filt_df_AEE_25 = filt_df_AEE_25[["idMatricula", "Id Escola", "Ano"]]
filt_df_AEE_24 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2024'] == 1)]
filt_df_AEE_24 = filt_df_AEE_24[["idMatricula", "Id Escola", "Ano"]]
filt_df_AEE_23 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2023'] == 1)]
filt_df_AEE_23 = filt_df_AEE_23[["idMatricula", "Id Escola", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_df_AEE_25, filt_df_AEE_24, filt_df_AEE_23]
combined_df_AEE = pd.concat([contar_distintos(df) for df in dataframes])

combined_df_AEE.rename(columns={'idMatricula': 'Nº de matrículas da educação especial (AEE)'}, inplace=True)
combined_df_AEE['Ano'] = combined_df_AEE['Ano'].astype(int)
display(combined_df_AEE)

,Id Escola,Ano,Nº de matrículas da educação especial (AEE)
0,343,2024,2


348 - Indicadores - Nº de matrículas transferidas da rede estadual para a rede municipal - ok - A Validar

In [78]:
# Tabela filtrada
tabela_clientes_E_M = tabela_clientes[["idMatricula", "Id Escola", "Ano", "NomeEtapa", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

Etapa_filtrar = [
    "Ensino fundamental de 9 anos - 5º Ano",
    "Ensino fundamental de 9 anos - 6º Ano",
    "Ensino fundamental de 9 anos - 7º Ano",
    "Ensino fundamental de 9 anos - 8º Ano",
    "Ensino fundamental de 9 anos - 9º Ano"
]

# Criar uma expressão regular que combine qualquer um dos textos
pattern = '|'.join(Etapa_filtrar)

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filt_df_E_M_25 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2025'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_25 = filt_df_E_M_25[["idMatricula", "Id Escola", "Ano"]]
filt_df_E_M_24 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2024'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_24 = filt_df_E_M_24[["idMatricula", "Id Escola", "Ano"]]
filt_df_E_M_23 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2023'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_23 = filt_df_E_M_23[["idMatricula", "Id Escola", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Id Escola', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_df_E_M_25, filt_df_E_M_24, filt_df_E_M_23]
combined_df_E_M = pd.concat([contar_distintos(df) for df in dataframes])

# Exibir o DataFrame combinado
combined_df_E_M = combined_df_E_M.groupby(['Id Escola', 'Ano'])['idMatricula'].sum().reset_index()
combined_df_E_M['Ano'] = combined_df_E_M['Ano'].astype(int)

# Ordena o DataFrame pelo ano (caso não esteja ordenado)
combined_df_E_M = combined_df_E_M.sort_values(by='Ano')

# Calcula a diferença de matrículas ano a ano
combined_df_E_M['Diferença de matrículas'] = combined_df_E_M['idMatricula'].diff()
combined_df_E_M.rename(columns={'Diferença de matrículas': 'Nº de matrículas transferidas da rede estadual para a rede municipal'}, inplace=True)
combined_df_E_M = combined_df_E_M[['Id Escola', 'Ano', 'Nº de matrículas transferidas da rede estadual para a rede municipal']]
display(combined_df_E_M)

,Id Escola,Ano,Nº de matrículas transferidas da rede estadual para a rede municipal
0,30,2024,NaN
30,778,2024,0.0
29,774,2024,0.0
28,738,2024,0.0
27,736,2024,33.0
26,725,2024,-33.0
25,681,2024,0.0
24,642,2024,194.0
23,636,2024,-174.0
22,599,2024,-18.0


Mesclagem de DateFrame Parcial

In [79]:
# Mesclar os DataFrames para adicionar as colunas com o 
# 159 - 2213
final_combined_df = combined_df.merge(combined_df_profissional, on=['Id Escola', 'Ano'], how='left')
#67
final_combined_df = final_combined_df.merge(combined_df_EJA, on=['Id Escola', 'Ano'], how='left')
#2246
final_combined_df = final_combined_df.merge(combined_df_EM, on=['Id Escola', 'Ano'], how='left')
#2208
final_combined_df = final_combined_df.merge(combined_df_cursos_profissional, on=['Id Escola', 'Ano'], how='left')
#2232
final_combined_df = final_combined_df.merge(combined_df_AEE, on=['Id Escola', 'Ano'], how='left')
#72
final_combined_df = final_combined_df.merge(combined_df_SEDUCTEC, on=['Id Escola', 'Ano'], how='left')
#2192
final_combined_df = final_combined_df.merge(EJA_EPT, on=['Id Escola', 'Ano'], how='left')
#348
final_combined_df = final_combined_df.merge(combined_df_E_M, on=['Id Escola', 'Ano'], how='left')
#65
final_combined_df = final_combined_df.merge(EPT, on=['Id Escola', 'Ano'], how='left')
#2196
final_combined_df = final_combined_df.merge(matricula_aut_indigena, on=['Id Escola', 'Ano'], how='left')
#2215
final_combined_df = final_combined_df.merge(matricula_aut_pretos, on=['Id Escola', 'Ano'], how='left')
#2238
final_combined_df = final_combined_df.merge(base_nota_media_M, on=['Id Escola', 'Ano'], how='left')
#402
final_combined_df = final_combined_df.merge(base_nota_media_LP, on=['Id Escola', 'Ano'], how='left')
#1194
final_combined_df = final_combined_df.merge(nota_LP, on=['Id Escola', 'Ano'], how='left')
#1193
final_combined_df = final_combined_df.merge(mat_M, on=['Id Escola', 'Ano'], how='left')
#163
final_combined_df = final_combined_df.merge(EM_geral, on=['Id Escola', 'Ano'], how='left')
#2237
final_combined_df = final_combined_df.merge(nota_media_LP, on=['Id Escola', 'Ano'], how='left')
#381
final_combined_df = final_combined_df.merge(base_nota_media_ITA_IME, on=['Id Escola', 'Ano'], how='left')
#382
final_combined_df = final_combined_df.merge(base_nota_media_esp_ITA_IME, on=['Id Escola', 'Ano'], how='left')
#385
final_combined_df = final_combined_df.merge(base_nota_freq, on=['Id Escola', 'Ano'], how='left')
#2202
final_combined_df = final_combined_df.merge(resultado_final, on=['Id Escola', 'Ano'], how='left')
#2212
final_combined_df = final_combined_df.merge(combined_df_EJA_EPT, on=['Id Escola', 'Ano'], how='left')
#2230
final_combined_df = final_combined_df.merge(combined_e_EM, on=['Id Escola', 'Ano'], how='left')
#63
final_combined_df = final_combined_df.merge(result_TI_EPT, on=['Id Escola', 'Ano'], how='left')
#153
final_combined_df = final_combined_df.merge(resultado_mun, on=['Id Escola', 'Ano'], how='left')
#61
final_combined_df = final_combined_df.merge(result_TI_EM, on=['Id Escola', 'Ano'], how='left')
#85
final_combined_df = final_combined_df.merge(freq_aprovativa, on=['Id Escola', 'Ano'], how='left')
#1191
final_combined_df = final_combined_df.merge(rural_final, on=['Id Escola', 'Ano'], how='left')
"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"
# Mudando o tipo da coluna idEscola para object
final_combined_df['Id Escola'] = final_combined_df['Id Escola'].astype(int).astype(str)

# Substituindo os NaN por 0 (ou outro valor apropriado)
#2230
final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'] = final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'].fillna(0)
#2212
final_combined_df['Nº de matrículas de EJA integrado ao EPT'] = final_combined_df['Nº de matrículas de EJA integrado ao EPT'].fillna(0)
#2202
final_combined_df['% de matrículas de tempo integral no ensino regular, na rede estadual'] = final_combined_df['% de matrículas de tempo integral no ensino regular, na rede estadual'].fillna(0)
#1194
final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'] = final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'].fillna(0)
#1193
final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'] = final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'].fillna(0)
#381
final_combined_df['Média nas avaliações nas demais disciplinas'] = final_combined_df['Média nas avaliações nas demais disciplinas'].fillna(0)
#163
final_combined_df['% de aprovação dos alunos do Ensino Médio'] = final_combined_df['% de aprovação dos alunos do Ensino Médio'].fillna(0)
#385
final_combined_df['% frequência dos estudantes inscritos nas turmas ITA/IME'] = final_combined_df['% frequência dos estudantes inscritos nas turmas ITA/IME'].fillna(0)
#382
final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'] = final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'].fillna(0)
#402
final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'].fillna(0)
#2237
final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'].fillna(0)
#2238
final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'].fillna(0)
#2215
final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'].fillna(0)
#2196
final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'].fillna(0)
#65
final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'] = final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'].fillna(0)
#348
final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'] = final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'].fillna(0)
#2192
final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'] = final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'].fillna(0)
#72
final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'] = final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'].fillna(0)
#2232
final_combined_df['Nº de matrículas da educação especial (AEE)'] = final_combined_df['Nº de matrículas da educação especial (AEE)'].fillna(0)
#2213
final_combined_df['Nº de matrículas EPT'] = final_combined_df['Nº de matrículas EPT'].fillna(0)
#67
final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'] = final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'].fillna(0)
#2246
final_combined_df['Nº de matrículas do Ensino Médio'] = final_combined_df['Nº de matrículas do Ensino Médio'].fillna(0)
#2208
final_combined_df['Nº de cursos EPT ofertados'] = final_combined_df['Nº de cursos EPT ofertados'].fillna(0)
#153
final_combined_df['% de municípios com escolas da rede estadual em tempo integral'] = final_combined_df['% de municípios com escolas da rede estadual em tempo integral'].fillna(0)
#61
final_combined_df['% de matrículas do ensino médio em tempo integral'] = final_combined_df['% de matrículas do ensino médio em tempo integral'].fillna(0)
#1191
final_combined_df['% de escolas da zona rural com oferta de EPT'] = final_combined_df['% de escolas da zona rural com oferta de EPT'].fillna(0)
#85
final_combined_df['% de estudantes da rede estadual com frequência regular em 75%'] = final_combined_df['% de estudantes da rede estadual com frequência regular em 75%'].fillna(0)
#63
final_combined_df['% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)'] = final_combined_df['% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)'].fillna(0)
"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"
#2230
final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'] = final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'].astype(int)
#2212
final_combined_df['Nº de matrículas de EJA integrado ao EPT'] = final_combined_df['Nº de matrículas de EJA integrado ao EPT'].astype(int)
#2213
final_combined_df['Nº de matrículas EPT'] = final_combined_df['Nº de matrículas EPT'].astype(int)
#67
final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'] = final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'].astype(int)
#2246
final_combined_df['Nº de matrículas do Ensino Médio'] = final_combined_df['Nº de matrículas do Ensino Médio'].astype(int)
#2208
final_combined_df['Nº de cursos EPT ofertados'] = final_combined_df['Nº de cursos EPT ofertados'].astype(int)
#2232
final_combined_df['Nº de matrículas da educação especial (AEE)'] = final_combined_df['Nº de matrículas da educação especial (AEE)'].astype(float).astype(int)
#72
final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'] = final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'].astype(float).astype(int)
#2192
final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'] = final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'].astype(float).astype(int)
#348
final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'] = final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'].astype(float).astype(int)
#65
final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'] = final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'].astype(float).astype(int)
#2196
final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'].astype(float).astype(int)
#2215
final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'].astype(float).astype(int)
#2238
final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'].astype(float).astype(int)
#2237
final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'].astype(float).astype(int)
#402
final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'].astype(float).astype(int)
#382
final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'] = final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'].astype(float).astype(int)
#381
final_combined_df['Média nas avaliações nas demais disciplinas'] = final_combined_df['Média nas avaliações nas demais disciplinas'].astype(float).astype(int)

final_combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 31 columns):
 #   Column                                                                                      Non-Null Count  Dtype  
---  ------                                                                                      --------------  -----  
 0   Id Escola                                                                                   372 non-null    object 
 1   Ano                                                                                         372 non-null    int32  
 2   Nº total de matrículas da rede estadual de educação                                         372 non-null    int64  
 3   Nº de matrículas EPT                                                                        372 non-null    int32  
 4   Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais                 372 non-null    int32  
 5   Nº de matrículas do Ensino Médio           

66 - Indicador - % de matrículas de EPT sobre o total de matrículas - ok - A Validar

Mesclagem de DateFrame Parcial

In [80]:
# Realizar o merge entre os dois DataFrames com base nas colunas 'Id Escola' e 'Ano'
final_percent_df = pd.merge(combined_df_profissional, combined_df, on=['Id Escola', 'Ano'])

# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas de EPT sobre o total de matrículas'] = final_percent_df['Nº de matrículas EPT'] / final_percent_df['Nº total de matrículas da rede estadual de educação']
final_percent_df['% de matrículas de EPT sobre o total de matrículas'] = final_percent_df['% de matrículas de EPT sobre o total de matrículas'].fillna(0)
final_percent_df['% de matrículas de EPT sobre o total de matrículas'] = final_percent_df['% de matrículas de EPT sobre o total de matrículas'].apply(lambda x: f"{x:.4f}").astype(float)

final_percent_df['Id Escola'] = final_percent_df['Id Escola'].astype(str)

final_percent_df = final_percent_df[["Id Escola", "Ano", "% de matrículas de EPT sobre o total de matrículas"]]

# Mesclagem para DF final
final_percent_df = final_combined_df.merge(final_percent_df, on=['Id Escola', 'Ano'], how='left')

# Exibir o DataFrame final combinado
print("Resultado final combinado:")
display(final_percent_df)

Resultado final combinado:


,Id Escola,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,% frequência dos estudantes inscritos nas turmas ITA/IME,"% de matrículas de tempo integral no ensino regular, na rede estadual",Nº de matrículas de EJA integrado ao EPT,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de matrículas de EPT sobre o total de matrículas
0,4,2024,25,0,15,0,0,0,0,0,...,0.0,1.0,0,0,0.0,1.0,0.0,0.666667,0.0,NaN
1,6,2024,8,0,0,8,0,0,1,0,...,0.0,0.0,0,0,0.0,0.0,0.0,0.285714,0.0,NaN
2,7,2024,7,0,7,0,0,0,1,0,...,0.0,0.0,0,0,0.0,0.0,0.0,1.000000,0.0,NaN
3,8,2024,30,0,0,0,0,0,30,0,...,0.0,0.0,0,0,0.0,0.0,0.0,1.000000,0.0,NaN
4,9,2024,61,0,0,0,0,0,38,0,...,0.0,0.0,0,0,0.0,0.0,0.0,0.842105,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367,1020,2024,29,0,0,1,0,0,28,0,...,0.0,0.0,0,1,0.0,0.0,1.0,0.000000,0.0,NaN
368,1029,2024,52,0,52,0,0,0,0,0,...,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0.0,NaN
369,1071,2024,51,0,51,0,0,0,0,0,...,0.0,0.0,0,0,0.0,0.0,0.0,0.724138,0.0,NaN
370,1089,2024,79,0,0,33,0,0,33,0,...,0.0,0.0,0,0,0.0,0.0,0.0,0.924051,0.0,NaN


2266 - Indicador - % de matrículas de EJA integrado ao EPT (rede estadual)

In [81]:
# Criar uma nova coluna com a divisão das colunas 'Nº de matrículas de EJA integrado ao EPT' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas de EJA integrado ao EPT (rede estadual)'] = final_percent_df['Nº de matrículas de EJA integrado ao EPT'] / final_percent_df['Nº total de matrículas da rede estadual de educação']

display(final_percent_df)

,Id Escola,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,"% de matrículas de tempo integral no ensino regular, na rede estadual",Nº de matrículas de EJA integrado ao EPT,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de matrículas de EPT sobre o total de matrículas,% de matrículas de EJA integrado ao EPT (rede estadual)
0,4,2024,25,0,15,0,0,0,0,0,...,1.0,0,0,0.0,1.0,0.0,0.666667,0.0,NaN,0.0
1,6,2024,8,0,0,8,0,0,1,0,...,0.0,0,0,0.0,0.0,0.0,0.285714,0.0,NaN,0.0
2,7,2024,7,0,7,0,0,0,1,0,...,0.0,0,0,0.0,0.0,0.0,1.000000,0.0,NaN,0.0
3,8,2024,30,0,0,0,0,0,30,0,...,0.0,0,0,0.0,0.0,0.0,1.000000,0.0,NaN,0.0
4,9,2024,61,0,0,0,0,0,38,0,...,0.0,0,0,0.0,0.0,0.0,0.842105,0.0,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367,1020,2024,29,0,0,1,0,0,28,0,...,0.0,0,1,0.0,0.0,1.0,0.000000,0.0,NaN,0.0
368,1029,2024,52,0,52,0,0,0,0,0,...,0.0,0,0,0.0,0.0,0.0,0.000000,0.0,NaN,0.0
369,1071,2024,51,0,51,0,0,0,0,0,...,0.0,0,0,0.0,0.0,0.0,0.724138,0.0,NaN,0.0
370,1089,2024,79,0,0,33,0,0,33,0,...,0.0,0,0,0.0,0.0,0.0,0.924051,0.0,NaN,0.0


2207 - Indicadores - % de matrículas EPT no EM sobre o total de matrículas - ok - A Validar 

In [82]:
# Criar uma nova coluna com a divisão das colunas 'Nº de matrículas de EJA integrado ao EPT' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas EPT no EM sobre o total de matrículas'] = final_percent_df['Nº de matrículas do Ensino Médio'] / final_percent_df['Nº total de matrículas da rede estadual de educação']

final_percent_df['% de matrículas EPT no EM sobre o total de matrículas'] = final_percent_df['% de matrículas de EPT sobre o total de matrículas'].apply(lambda x: f"{x:.4f}").astype(float)

display(final_percent_df)

,Id Escola,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,Nº de matrículas de EJA integrado ao EPT,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de matrículas de EPT sobre o total de matrículas,% de matrículas de EJA integrado ao EPT (rede estadual),% de matrículas EPT no EM sobre o total de matrículas
0,4,2024,25,0,15,0,0,0,0,0,...,0,0,0.0,1.0,0.0,0.666667,0.0,NaN,0.0,NaN
1,6,2024,8,0,0,8,0,0,1,0,...,0,0,0.0,0.0,0.0,0.285714,0.0,NaN,0.0,NaN
2,7,2024,7,0,7,0,0,0,1,0,...,0,0,0.0,0.0,0.0,1.000000,0.0,NaN,0.0,NaN
3,8,2024,30,0,0,0,0,0,30,0,...,0,0,0.0,0.0,0.0,1.000000,0.0,NaN,0.0,NaN
4,9,2024,61,0,0,0,0,0,38,0,...,0,0,0.0,0.0,0.0,0.842105,0.0,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367,1020,2024,29,0,0,1,0,0,28,0,...,0,1,0.0,0.0,1.0,0.000000,0.0,NaN,0.0,NaN
368,1029,2024,52,0,52,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.000000,0.0,NaN,0.0,NaN
369,1071,2024,51,0,51,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.724138,0.0,NaN,0.0,NaN
370,1089,2024,79,0,0,33,0,0,33,0,...,0,0,0.0,0.0,0.0,0.924051,0.0,NaN,0.0,NaN


Especificação Indicadores por Superintendência

In [83]:
# Tabela de indicadores estratégicos filtrado
especificacao_superintendencia_df = tabela_ind_estrategico[["CodigoUnidadeNegocio", "NomeUnidade", "SiglaUN", "Codigo Indicador", "Indicador", "CodigoProjeto", "NomeProjeto", "MetaAcumuladaAno", "MetaMes", "UnidadeMedida", "CasasDecimais", "Polaridade", "Periodicidade", "Fonte", "GlossarioIndicador", "Cod_Responsavel_Ind", "ResponsavelIndicador", "Cod_Responsavel_Coleta", "ResponsavelColeta", "TipoIndicadorEstPro", "Analise", "Ano_Meta", "Mes_Meta"]]

# Mesclar os DataFrames para adicionar as colunas com o cálculo
#especificacao_superintendência_df = especificacao_superintendência.merge(Tab_estrategico, on='NomeIndicador', how='left')

especificacao_superintendencia_df = especificacao_superintendencia_df.drop_duplicates()
especificacao_superintendencia_df['CodigoUnidadeNegocio'] = especificacao_superintendencia_df['CodigoUnidadeNegocio'].fillna(0)
especificacao_superintendencia_df['NomeUnidade'] = especificacao_superintendencia_df['NomeUnidade'].fillna('')
especificacao_superintendencia_df['SiglaUN'] = especificacao_superintendencia_df['SiglaUN'].fillna('')
especificacao_superintendencia_df['CodigoProjeto'] = especificacao_superintendencia_df['CodigoProjeto'].fillna(0)
especificacao_superintendencia_df['NomeProjeto'] = especificacao_superintendencia_df['NomeProjeto'].fillna('')
especificacao_superintendencia_df['UnidadeMedida'] = especificacao_superintendencia_df['UnidadeMedida'].fillna('')
especificacao_superintendencia_df['Polaridade'] = especificacao_superintendencia_df['Polaridade'].fillna('')
especificacao_superintendencia_df['ResponsavelIndicador'] = especificacao_superintendencia_df['ResponsavelIndicador'].fillna('')
especificacao_superintendencia_df['ResponsavelColeta'] = especificacao_superintendencia_df['ResponsavelColeta'].fillna('')
especificacao_superintendencia_df['Ano_Meta'] = especificacao_superintendencia_df['Ano_Meta'].fillna(0)
especificacao_superintendencia_df['Mes_Meta'] = especificacao_superintendencia_df['Mes_Meta'].fillna(0)
especificacao_superintendencia_df['MetaMes'] = especificacao_superintendencia_df['MetaMes'].fillna(np.nan)
especificacao_superintendencia_df['MetaAcumuladaAno'] = especificacao_superintendencia_df['MetaAcumuladaAno'].fillna(0)

especificacao_superintendencia_df['CodigoUnidadeNegocio'] = especificacao_superintendencia_df['CodigoUnidadeNegocio'].astype(int)
especificacao_superintendencia_df['CodigoProjeto'] = especificacao_superintendencia_df['CodigoProjeto'].astype(int)
especificacao_superintendencia_df['Ano_Meta'] = especificacao_superintendencia_df['Ano_Meta'].astype(int)
especificacao_superintendencia_df['Mes_Meta'] = especificacao_superintendencia_df['Mes_Meta'].astype(int)
especificacao_superintendencia_df['MetaMes'] = especificacao_superintendencia_df['MetaMes'].astype(float)
especificacao_superintendencia_df['MetaAcumuladaAno'] = especificacao_superintendencia_df['MetaAcumuladaAno'].astype(float)

%run Caminhos.ipynb

# Salvando a tabela especificacao_ind_projetos_df em um arquivo CSV
especificacao_superintendencia_df.to_csv(f'{pasta_hierarquia}{caminho_especificacao_superintendencia_df_csv}', index=False)

# salvar os dados em um arquivo binário
#especificacao_superintendencia_df.to_pickle(f'{pasta_hierarquia}{caminho_especificacao_superintendencia_df_pkl}')
"----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"
# Tabela de indicadores Projetos filtrado
especificacao_ind_projetos_df = tabela_indicadores_projetos[["CodigoUnidadeNegocio", "NomeUnidade", "SiglaUN", "Codigo Indicador", "Indicador", "CodigoProjeto", "NomeProjeto", "MetaAcumuladaAno", "MetaMes", "UnidadeMedida", "CasasDecimais", "Polaridade", "Periodicidade", "Fonte", "GlossarioIndicador", "Cod_Responsavel_Ind", "ResponsavelIndicador", "Cod_Responsavel_Coleta", "ResponsavelColeta", "TipoIndicadorEstPro", "Analise", "Ano_Meta", "Mes_Meta"]]

# Mesclar os DataFrames para adicionar as colunas com o cálculo
#especificacao_superintendência_df = especificacao_superintendência.merge(Tab_estrategico, on='NomeIndicador', how='left')

especificacao_ind_projetos_df = especificacao_ind_projetos_df.drop_duplicates()
especificacao_ind_projetos_df['CodigoUnidadeNegocio'] = especificacao_ind_projetos_df['CodigoUnidadeNegocio'].fillna(0)
especificacao_ind_projetos_df['NomeUnidade'] = especificacao_ind_projetos_df['NomeUnidade'].fillna('')
especificacao_ind_projetos_df['SiglaUN'] = especificacao_ind_projetos_df['SiglaUN'].fillna('')
especificacao_ind_projetos_df['CodigoProjeto'] = especificacao_ind_projetos_df['CodigoProjeto'].fillna(0)
especificacao_ind_projetos_df['NomeProjeto'] = especificacao_ind_projetos_df['NomeProjeto'].fillna('')
especificacao_ind_projetos_df['UnidadeMedida'] = especificacao_ind_projetos_df['UnidadeMedida'].fillna('')
especificacao_ind_projetos_df['Polaridade'] = especificacao_ind_projetos_df['Polaridade'].fillna('')
especificacao_ind_projetos_df['ResponsavelIndicador'] = especificacao_ind_projetos_df['ResponsavelIndicador'].fillna('')
especificacao_ind_projetos_df['ResponsavelColeta'] = especificacao_ind_projetos_df['ResponsavelColeta'].fillna('')
especificacao_ind_projetos_df['Ano_Meta'] = especificacao_ind_projetos_df['Ano_Meta'].fillna(0)
especificacao_ind_projetos_df['Mes_Meta'] = especificacao_ind_projetos_df['Mes_Meta'].fillna(0)
especificacao_ind_projetos_df['MetaMes'] = especificacao_ind_projetos_df['MetaMes'].fillna(np.nan)
especificacao_ind_projetos_df['MetaAcumuladaAno'] = especificacao_ind_projetos_df['MetaAcumuladaAno'].fillna(0)
especificacao_ind_projetos_df['Cod_Responsavel_Coleta'] = especificacao_ind_projetos_df['Cod_Responsavel_Coleta'].fillna(0)

especificacao_ind_projetos_df['CodigoUnidadeNegocio'] = especificacao_ind_projetos_df['CodigoUnidadeNegocio'].astype(int)
especificacao_ind_projetos_df['CodigoProjeto'] = especificacao_ind_projetos_df['CodigoProjeto'].astype(int)
especificacao_ind_projetos_df['Ano_Meta'] = especificacao_ind_projetos_df['Ano_Meta'].astype(int)
especificacao_ind_projetos_df['Mes_Meta'] = especificacao_ind_projetos_df['Mes_Meta'].astype(int)
especificacao_ind_projetos_df['MetaMes'] = especificacao_ind_projetos_df['MetaMes'].astype(float)
especificacao_ind_projetos_df['MetaAcumuladaAno'] = especificacao_ind_projetos_df['MetaAcumuladaAno'].astype(float)
especificacao_ind_projetos_df['Cod_Responsavel_Coleta'] = especificacao_ind_projetos_df['Cod_Responsavel_Coleta'].astype(int)

%run Caminhos.ipynb

# Salvando a tabela especificacao_ind_projetos_df em um arquivo CSV
especificacao_ind_projetos_df.to_csv(f'{pasta_hierarquia}{caminho_especificacao_ind_projetos_df_csv}', index=False)

# salvar os dados em um arquivo binário
#especificacao_ind_projetos_df.to_pickle(f'{pasta_hierarquia}{caminho_especificacao_ind_projetos_df_pkl}')

"------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"
# Acrescentando as linhas de especificacao_ind_projetos_df ao final de especificacao_superintendência_df
especificacao_ind_O_E = pd.concat([especificacao_superintendencia_df, especificacao_ind_projetos_df], ignore_index=True)

# Salvando a tabela especificacao_ind_projetos_df em um arquivo CSV
especificacao_ind_O_E.to_csv(f'{pasta_hierarquia}{caminho_especificacao_ind_O_E_csv}', index=False)
especificacao_ind_O_E.to_parquet(f'{pasta_hierarquia}{caminho_especificacao_ind_O_E_parquet}')
display(especificacao_ind_O_E)

Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/
Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/


,CodigoUnidadeNegocio,NomeUnidade,SiglaUN,Codigo Indicador,Indicador,CodigoProjeto,NomeProjeto,MetaAcumuladaAno,MetaMes,UnidadeMedida,...,Fonte,GlossarioIndicador,Cod_Responsavel_Ind,ResponsavelIndicador,Cod_Responsavel_Coleta,ResponsavelColeta,TipoIndicadorEstPro,Analise,Ano_Meta,Mes_Meta
0,451,Superintendência de Ensino,SUPEN,61,% de matrículas do ensino médio em tempo integral,447,Ser Integral faz a diferença,90.0,90.0,%,...,CENSO ESCOLAR,<p>Att A<br></p><p>Compromisso 4</p><p><br></p...,128,Viviane Faria,178,Elynne Barros,E,None,2026,12
1,451,Superintendência de Ensino,SUPEN,61,% de matrículas do ensino médio em tempo integral,447,Ser Integral faz a diferença,90.0,90.0,%,...,CENSO ESCOLAR,<p>Att A<br></p><p>Compromisso 4</p><p><br></p...,128,Viviane Faria,178,Elynne Barros,E,None,2026,11
2,451,Superintendência de Ensino,SUPEN,61,% de matrículas do ensino médio em tempo integral,447,Ser Integral faz a diferença,90.0,90.0,%,...,CENSO ESCOLAR,<p>Att A<br></p><p>Compromisso 4</p><p><br></p...,128,Viviane Faria,178,Elynne Barros,E,None,2026,10
3,451,Superintendência de Ensino,SUPEN,61,% de matrículas do ensino médio em tempo integral,447,Ser Integral faz a diferença,90.0,90.0,%,...,CENSO ESCOLAR,<p>Att A<br></p><p>Compromisso 4</p><p><br></p...,128,Viviane Faria,178,Elynne Barros,E,None,2026,9
4,451,Superintendência de Ensino,SUPEN,61,% de matrículas do ensino médio em tempo integral,447,Ser Integral faz a diferença,90.0,90.0,%,...,CENSO ESCOLAR,<p>Att A<br></p><p>Compromisso 4</p><p><br></p...,128,Viviane Faria,178,Elynne Barros,E,None,2026,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10260,473,Superintendência de Gestão Interna e Educação ...,SGI,1432,n° de bolsas pagas pelo Alfabetiza Piauí,556,Alfabetiza Piauí,70.0,70.0,Nº,...,Sistema Alfabetiza Piauí,<p>Att M</p><p>formula&nbsp;n° de bolsas pagas...,159,Gilmania Carvalho,197,Isamara Nascimento,O,None,2024,5
10261,473,Superintendência de Gestão Interna e Educação ...,SGI,1432,n° de bolsas pagas pelo Alfabetiza Piauí,556,Alfabetiza Piauí,70.0,70.0,Nº,...,Sistema Alfabetiza Piauí,<p>Att M</p><p>formula&nbsp;n° de bolsas pagas...,159,Gilmania Carvalho,197,Isamara Nascimento,O,None,2024,4
10262,473,Superintendência de Gestão Interna e Educação ...,SGI,1432,n° de bolsas pagas pelo Alfabetiza Piauí,556,Alfabetiza Piauí,70.0,70.0,Nº,...,Sistema Alfabetiza Piauí,<p>Att M</p><p>formula&nbsp;n° de bolsas pagas...,159,Gilmania Carvalho,197,Isamara Nascimento,O,None,2024,3
10263,473,Superintendência de Gestão Interna e Educação ...,SGI,1432,n° de bolsas pagas pelo Alfabetiza Piauí,556,Alfabetiza Piauí,70.0,70.0,Nº,...,Sistema Alfabetiza Piauí,<p>Att M</p><p>formula&nbsp;n° de bolsas pagas...,159,Gilmania Carvalho,197,Isamara Nascimento,O,None,2024,2


In [84]:
import pandas as pd

especificacao_ind_O_E_duplicate = especificacao_ind_O_E[["Codigo Indicador", "TipoIndicadorEstPro"]]

# Removendo duplicatas com base nas colunas selecionadas
especificacao_ind_O_E_duplicate = especificacao_ind_O_E_duplicate.drop_duplicates()

# Supondo que as colunas sejam "Codigo Indicador", "NomeUnidade", e "SiglaUN"
colunas_para_verificar = ['Codigo Indicador', 'TipoIndicadorEstPro']

# Verificando os duplicados com base nessas três colunas
duplicados = especificacao_ind_O_E_duplicate[especificacao_ind_O_E_duplicate.duplicated(subset=colunas_para_verificar, keep=False)]

# Analise pra gerar a nova
# Verificando os duplicados na coluna "Codigo Indicador"
duplicados = especificacao_ind_O_E_duplicate[especificacao_ind_O_E_duplicate.duplicated(subset=['Codigo Indicador'], keep=False)]

# Função para concatenar "Codigo Indicador" com "TipoIndicadorEstPro" se for duplicado
def concatena_codigo(row):
    if row['Codigo Indicador'] in duplicados['Codigo Indicador'].values:
        return f"{row['Codigo Indicador']}_{row['TipoIndicadorEstPro']}"
    else:
        return row['Codigo Indicador']

# Aplicando a função para criar a nova coluna
especificacao_ind_O_E_duplicate['CodigoConcatenado'] = especificacao_ind_O_E_duplicate.apply(concatena_codigo, axis=1)

# Exibindo o DataFrame atualizado
display(especificacao_ind_O_E_duplicate)

# Realizando a mesclagem com base nas colunas "Codigo Indicador" e "TipoIndicadorEstPro"
especificacao_ind_O_E = especificacao_ind_O_E.merge(
    especificacao_ind_O_E_duplicate[['Codigo Indicador', 'TipoIndicadorEstPro', 'CodigoConcatenado']],
    on=['Codigo Indicador', 'TipoIndicadorEstPro'],
    how='left'
)

# Salvando o DataFrame atualizado em CSV e Pickle
especificacao_ind_O_E.to_csv(f'{pasta_hierarquia}{caminho_especificacao_ind_O_E_csv}', index=False)
especificacao_ind_O_E.to_pickle(f'{pasta_hierarquia}{caminho_especificacao_ind_O_E_pkl}')
display(especificacao_ind_O_E)

,Codigo Indicador,TipoIndicadorEstPro,CodigoConcatenado
0,61,E,61
60,85,E,85
120,97,E,97
125,98,E,98
130,103,E,103
...,...,...,...
10085,337,O,337
10109,363,O,363
10157,1430,O,1430
10193,1431,O,1431


,CodigoUnidadeNegocio,NomeUnidade,SiglaUN,Codigo Indicador,Indicador,CodigoProjeto,NomeProjeto,MetaAcumuladaAno,MetaMes,UnidadeMedida,...,GlossarioIndicador,Cod_Responsavel_Ind,ResponsavelIndicador,Cod_Responsavel_Coleta,ResponsavelColeta,TipoIndicadorEstPro,Analise,Ano_Meta,Mes_Meta,CodigoConcatenado
0,451,Superintendência de Ensino,SUPEN,61,% de matrículas do ensino médio em tempo integral,447,Ser Integral faz a diferença,90.0,90.0,%,...,<p>Att A<br></p><p>Compromisso 4</p><p><br></p...,128,Viviane Faria,178,Elynne Barros,E,None,2026,12,61
1,451,Superintendência de Ensino,SUPEN,61,% de matrículas do ensino médio em tempo integral,447,Ser Integral faz a diferença,90.0,90.0,%,...,<p>Att A<br></p><p>Compromisso 4</p><p><br></p...,128,Viviane Faria,178,Elynne Barros,E,None,2026,11,61
2,451,Superintendência de Ensino,SUPEN,61,% de matrículas do ensino médio em tempo integral,447,Ser Integral faz a diferença,90.0,90.0,%,...,<p>Att A<br></p><p>Compromisso 4</p><p><br></p...,128,Viviane Faria,178,Elynne Barros,E,None,2026,10,61
3,451,Superintendência de Ensino,SUPEN,61,% de matrículas do ensino médio em tempo integral,447,Ser Integral faz a diferença,90.0,90.0,%,...,<p>Att A<br></p><p>Compromisso 4</p><p><br></p...,128,Viviane Faria,178,Elynne Barros,E,None,2026,9,61
4,451,Superintendência de Ensino,SUPEN,61,% de matrículas do ensino médio em tempo integral,447,Ser Integral faz a diferença,90.0,90.0,%,...,<p>Att A<br></p><p>Compromisso 4</p><p><br></p...,128,Viviane Faria,178,Elynne Barros,E,None,2026,8,61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10260,473,Superintendência de Gestão Interna e Educação ...,SGI,1432,n° de bolsas pagas pelo Alfabetiza Piauí,556,Alfabetiza Piauí,70.0,70.0,Nº,...,<p>Att M</p><p>formula&nbsp;n° de bolsas pagas...,159,Gilmania Carvalho,197,Isamara Nascimento,O,None,2024,5,1432
10261,473,Superintendência de Gestão Interna e Educação ...,SGI,1432,n° de bolsas pagas pelo Alfabetiza Piauí,556,Alfabetiza Piauí,70.0,70.0,Nº,...,<p>Att M</p><p>formula&nbsp;n° de bolsas pagas...,159,Gilmania Carvalho,197,Isamara Nascimento,O,None,2024,4,1432
10262,473,Superintendência de Gestão Interna e Educação ...,SGI,1432,n° de bolsas pagas pelo Alfabetiza Piauí,556,Alfabetiza Piauí,70.0,70.0,Nº,...,<p>Att M</p><p>formula&nbsp;n° de bolsas pagas...,159,Gilmania Carvalho,197,Isamara Nascimento,O,None,2024,3,1432
10263,473,Superintendência de Gestão Interna e Educação ...,SGI,1432,n° de bolsas pagas pelo Alfabetiza Piauí,556,Alfabetiza Piauí,70.0,70.0,Nº,...,<p>Att M</p><p>formula&nbsp;n° de bolsas pagas...,159,Gilmania Carvalho,197,Isamara Nascimento,O,None,2024,2,1432


Pivot Resultados 

In [85]:
final_percent_df['% de matrículas de EPT sobre o total de matrículas'] = final_percent_df['% de matrículas de EPT sobre o total de matrículas'].replace('.', ',')

# Selecionar automaticamente todas as colunas para value_vars, exceto as que estão em id_vars
id_vars = ['Id Escola', 'Ano']
value_vars = [col for col in final_percent_df.columns if col not in id_vars]

# Transformar o DataFrame para a forma PIVOT
long_df = final_percent_df.melt(id_vars=id_vars, 
                                 value_vars=value_vars,
                                 var_name='Indicador', value_name='Quantidade')

# Pivotar para obter as colunas separadas para os anos 2023 e 2024
pivot_df = long_df.pivot_table(index=['Id Escola', 'Indicador', 'Ano'], values='Quantidade').reset_index()


# Renomear as colunas para torná-las mais legíveis
pivot_df.columns.name = None
pivot_df.rename(columns={2023: '2023', 2024: '2024'}, inplace=True)

# Seleciona as colunas "CasasDecimais" e "CodigoIndicador" do dataframe especificacao_superintendência_df
resumo_calculado = especificacao_ind_O_E[['Codigo Indicador', 'Indicador', 'CasasDecimais', 'UnidadeMedida', 'TipoIndicadorEstPro']]

# Mesclar os DataFrames para adicionar as colunas com o cálculo
pivot_df = resumo_calculado.merge(pivot_df, on='Indicador', how='left')

pivot_df['CasasDecimais'] = pivot_df['CasasDecimais'].fillna(0)
pivot_df['CasasDecimais'] = pivot_df['CasasDecimais'].astype(int)
pivot_df['Codigo Indicador'] = pivot_df['Codigo Indicador'].fillna(0)
pivot_df['Codigo Indicador'] = pivot_df['Codigo Indicador'].astype(str)
pivot_df['Ano'] = pivot_df['Ano'].fillna(0)
pivot_df['Ano'] = pivot_df['Ano'].astype(int)

# Função para formatar o valor conforme a quantidade de casas decimais e tipo de unidade
#def formatar_quantidade(row):
#    casas_decimais = int(row['CasasDecimais'])
#    if row['UnidadeMedida'] == '%':
        # Para percentuais, primeiro remover '%' se presente e converter para float
#        valor = row['Quantidade']
#        return f"{valor:.{casas_decimais}f}".replace('.', ',')
#    else:
#        # Para números, formatar diretamente com as casas decimais
#        valor = row['Quantidade']
#        return f"{valor:.{casas_decimais}f}".replace('.', ',')

# Formatar a coluna 'Quantidade'
#pivot_df['Quantidade'] = pivot_df.apply(formatar_quantidade, axis=1)
pivot_df = pivot_df[['Id Escola', 'Codigo Indicador', 'Ano', 'Quantidade', 'TipoIndicadorEstPro']]

## IDENTIFICAR MOTIVO DE DUPLICAÇÃO DE LINHAS DO DATAFRAME
pivot_df = pivot_df.drop_duplicates(keep='first')

from datetime import datetime

# Capturar a data e hora atuais
data_hora_execucao = datetime.now().strftime('%d/%m/%Y %H:%M:%S')

# Adicionar uma nova coluna ao DataFrame com a data e hora da última execução do script
pivot_df['Data Cadastro'] = data_hora_execucao

# Converter a coluna DataHoraExecucao para datetime e extrair o mês
pivot_df['Mes_Referente'] = pd.to_datetime(pivot_df['Data Cadastro'], format='%d/%m/%Y %H:%M:%S').dt.month

pivot_df['Mes_Referente'] = pivot_df['Mes_Referente'].astype(int)

pivot_df = pivot_df[['Id Escola', 'Codigo Indicador', 'Ano', 'Quantidade', 'Data Cadastro', 'Mes_Referente', 'TipoIndicadorEstPro']]

# Exibindo a tabela final pivotada
print("Tabela Final Pivotada:")
display(pivot_df)
pivot_df.info()

Tabela Final Pivotada:


,Id Escola,Codigo Indicador,Ano,Quantidade,Data Cadastro,Mes_Referente,TipoIndicadorEstPro
0,10,61,2024,0.0,17/06/2025 14:25:11,6,E
1,1008,61,2024,0.0,17/06/2025 14:25:11,6,E
2,101,61,2024,0.0,17/06/2025 14:25:11,6,E
3,102,61,2024,0.0,17/06/2025 14:25:11,6,E
4,1020,61,2024,1.0,17/06/2025 14:25:11,6,E
...,...,...,...,...,...,...,...
498370,NaN,337,0,NaN,17/06/2025 14:25:11,6,O
498394,NaN,363,0,NaN,17/06/2025 14:25:11,6,O
498442,NaN,1430,0,NaN,17/06/2025 14:25:11,6,O
498478,NaN,1431,0,NaN,17/06/2025 14:25:11,6,O


<class 'pandas.core.frame.DataFrame'>
Index: 9563 entries, 0 to 498514
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id Escola            9336 non-null   object 
 1   Codigo Indicador     9563 non-null   object 
 2   Ano                  9563 non-null   int32  
 3   Quantidade           9336 non-null   float64
 4   Data Cadastro        9563 non-null   object 
 5   Mes_Referente        9563 non-null   int32  
 6   TipoIndicadorEstPro  9563 non-null   object 
dtypes: float64(1), int32(2), object(4)
memory usage: 523.0+ KB


Preparação e definição de Base

In [86]:
# Preencher valores NaN na coluna 'Hierarquia' com string vazia
tabela_base_gerada['Hierarquia'] = tabela_base_gerada['Hierarquia'].fillna('')

# Filtro por escola
BASE_GERADA = tabela_base_gerada[(tabela_base_gerada['Hierarquia'].str.contains("Escola"))]

# Adicionar a nova coluna 'TipoIndicadorEstPro' com a informação 'Manual' em todas as linhas

BASE_GERADA = BASE_GERADA[['Id Escola', 'Codigo Indicador', 'Ano', 'Quantidade', 'Data Cadastro', 'Mes_Referente', 'TipoIndicadorEstPro']]

BASE_GERADA['Id Escola'] = BASE_GERADA['Id Escola'].fillna(0)
BASE_GERADA['Id Escola'] = BASE_GERADA['Id Escola'].astype(str)

# Exibir o DataFrame
display(BASE_GERADA)
BASE_GERADA.info()

,Id Escola,Codigo Indicador,Ano,Quantidade,Data Cadastro,Mes_Referente,TipoIndicadorEstPro
2353,5.0,65,2024,8.0,2024-08-23 09:23:38,8,E
2354,3.0,65,2024,99.0,2024-08-26 10:18:15,8,E
2355,5.0,65,2024,8.0,2024-08-23 09:23:38,8,E
2356,3.0,65,2024,99.0,2024-08-26 10:18:15,8,E
2390,5.0,65,2024,8.0,2024-08-23 09:23:38,8,E
...,...,...,...,...,...,...,...
19715,730.0,2281,2024,1.0,2025-02-14 10:17:55,11,E
19716,737.0,2281,2024,1.0,2025-02-14 10:17:55,11,E
19717,757.0,2281,2024,1.0,2025-02-14 10:17:55,11,E
19718,805.0,2281,2024,1.0,2025-02-14 10:17:55,11,E


<class 'pandas.core.frame.DataFrame'>
Index: 539 entries, 2353 to 19719
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Id Escola            539 non-null    object        
 1   Codigo Indicador     539 non-null    object        
 2   Ano                  539 non-null    int64         
 3   Quantidade           539 non-null    float64       
 4   Data Cadastro        539 non-null    datetime64[ns]
 5   Mes_Referente        539 non-null    int64         
 6   TipoIndicadorEstPro  539 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 33.7+ KB


In [87]:
# Mesclar os DataFrames para adicionar as colunas com o cálculo
base_indicadores = pd.concat([BASE_GERADA, pivot_df], ignore_index=True)
display(base_indicadores)

,Id Escola,Codigo Indicador,Ano,Quantidade,Data Cadastro,Mes_Referente,TipoIndicadorEstPro
0,5.0,65,2024,8.0,2024-08-23 09:23:38,8,E
1,3.0,65,2024,99.0,2024-08-26 10:18:15,8,E
2,5.0,65,2024,8.0,2024-08-23 09:23:38,8,E
3,3.0,65,2024,99.0,2024-08-26 10:18:15,8,E
4,5.0,65,2024,8.0,2024-08-23 09:23:38,8,E
...,...,...,...,...,...,...,...
10097,NaN,337,0,NaN,17/06/2025 14:25:11,6,O
10098,NaN,363,0,NaN,17/06/2025 14:25:11,6,O
10099,NaN,1430,0,NaN,17/06/2025 14:25:11,6,O
10100,NaN,1431,0,NaN,17/06/2025 14:25:11,6,O


In [88]:
# O script atualiza diariamente um arquivo CSV acumulando dados e, no primeiro dia de cada mês, cria um backup mensal com os dados do último dia do mês anterior.

from datetime import datetime, timedelta
import os
import shutil

# Verificar se a pasta historico_diario existe, se não, criar
#historico_diario_path = 'historico_diario'
%run Caminhos.ipynb

# Caminhos para as pastas de histórico diário e mensal
historico_diario_path = os.path.join(pasta_hierarquia_historico, 'historico_diario')
historico_mensal_path = os.path.join(pasta_hierarquia_historico, 'historico_mensal')

# Verificar se a pasta historico_diario existe, se não, criar
if not os.path.exists(historico_diario_path):
    os.makedirs(historico_diario_path)

# Nome do arquivo CSV diário
arquivo_diario = os.path.join(historico_diario_path, 'dados_diario_escola.csv')

# Verificar se o arquivo já existe
if os.path.exists(arquivo_diario):
    # Se o arquivo já existir, abrir em modo de apêndice e sem o cabeçalho
    base_indicadores.to_csv(arquivo_diario, mode='a', header=False, index=False)
else:
    # Se o arquivo não existir, criar um novo com o cabeçalho
    base_indicadores.to_csv(arquivo_diario, index=False)

# Verificar se hoje é o primeiro dia do mês
hoje = datetime.now()
if hoje.day == 1:
    # Obter a data do último dia do mês anterior
    ultimo_dia_mes_anterior = hoje.replace(day=1) - timedelta(days=1)
    
    # Verificar se o arquivo diário tem entradas para o último dia do mês anterior
    dados_diarios = pd.read_csv(arquivo_diario)
    dados_ultimo_dia = dados_diarios[dados_diarios['Data Cadastro'].str.startswith(ultimo_dia_mes_anterior.strftime('%Y-%m-%d'))]

    if not dados_ultimo_dia.empty:
        # Verificar se a pasta historico_mensal existe, se não, criar
        if not os.path.exists(historico_mensal_path):
            os.makedirs(historico_mensal_path)
        
        # Salvar os dados do último dia do mês anterior no arquivo mensal
        dados_ultimo_dia.to_csv(os.path.join(historico_mensal_path, f'dados_escola_{ultimo_dia_mes_anterior.strftime("%Y-%m-%d")}.csv'), index=False)


Erro ao conectar: ORA-12545: Connect failed because target host or object does not exist
Help: https://docs.oracle.com/error-help/db/ora-12545/


OSError: [Errno 22] Invalid argument